# Will this company report a loss next quarter? A leakage-safe forecast from SEC filings

**Question (S1).** Given a company's latest 10-K or 10-Q, how likely is it to report a net loss in its **next quarter**? A second view, the **transition**: among companies that are profitable *now*, which ones turn to a loss?

**Data.** The SEC's Financial Statement Data Sets (DERA), pivoted to one row per filing, in 14 quarterly files (2023Q1 to 2026Q2): the dataset [`sec-10k-10q-wide-v3`](https://www.kaggle.com/datasets/siemmebrahtu/sec-10k-10q-wide-v3), ETL version 3.2. About 81,000 filings from more than 6,000 companies.

**Headline results (test period: label quarters Oct 2025 to May 2026, 10,530 filings).** Cell 17 reprints these numbers from the code, Cell 19 gives each one a 95% confidence interval, and Cell 18 repeats the test under a stricter, filing-date protocol.

| | Test ROC-AUC | Test PR-AUC |
|---|---|---|
| Persistence rule ("next quarter looks like this one") | 0.827 | 0.755 |
| History table (last two quarters) | 0.875 | 0.822 |
| Logistic regression | 0.911 | 0.890 |
| **Gradient boosting** | **0.924** | **0.911** |

- The model beats persistence in **7 of 7** rolling test quarters (mean ROC-AUC lift +0.10, Cell 12b).
- In the transition view (profitable now), it ranks the 17% that turn to a loss with ROC-AUC 0.85, where persistence has no ranking power at all.
- The probabilities are well calibrated (Cell 9).
- **Where it fails:** private-credit funds and BDCs. Their loss rate is normally 3-11% per quarter but jumped to 41% in 2026Q1, a shock shared across the group that their own filings do not foreshadow. On them the model scores ROC-AUC 0.61 (Cell 12d).

**Part 2 in brief (Cells 14 to 17, same split and rules)**
- **U1, company archetypes:** k-means, fitted on training rows only, finds 7 stable company types, from established revenue businesses (38% next-quarter loss rate) to deep-loss micro companies (90%). 86% of filings keep their company's type from the previous quarter. As a model feature the archetype adds only +0.0001 test ROC-AUC, so it is kept as a *description* of company types, not as an input. Cell 14 shows why seven, and not the ten the silhouette alone would suggest.
- **A cheaper model:** gradient boosting on just the top 8 features keeps **95%** of the lift over persistence (test ROC-AUC 0.919 vs 0.924). A logistic regression on the same 8 features keeps only 36%, because those features were ranked by the boosting model and leave out the loss flags a linear model needs.
- **S2, equity turns negative (exploratory):** among companies with positive equity now (a 1.6% event rate in the test period), the model reaches ROC-AUC **0.959** and PR-AUC **0.323** against 0.263 for the best one-line rule ("fewest quarters of losses the equity can absorb"). It finds **89%** of the events in the riskiest 10% of filings and beats the rule in 6 of 7 rolling quarters. With only 141 test events, the PR-AUC lead over the rule is not statistically established (95% interval −0.011 to +0.131, Cell 19): S2 is a promising exploratory screening model, not a proven improvement.

**Part 3 in brief (Cells 18 to 22): how far to trust the results, and how to use them**
- **Filing-date validation (Cell 18):** under the label-quarter split, 17% of training answers were not yet public when the first validation forecasts were made. Re-run so that the model only learns from answers already public on the day of each forecast, the scores barely move (validation 0.933 vs 0.934, test 0.925 vs 0.925), and the model beats persistence in all 8 filing quarters.
- **Confidence intervals (Cell 19):** a company-clustered bootstrap. Test ROC-AUC 0.924 (95% interval 0.918 to 0.929); lift over persistence +0.098 (+0.093 to +0.103).
- **Decision thresholds (Cell 20):** cut-offs chosen on validation data only, then checked on test. The S1 default (0.424) keeps precision at 0.82 and recall at 0.87 on test; the S2 default (0.030) catches 82% of negative-equity events by flagging 6.8% of filings.
- **Later-period and unseen-company tests (Cell 21):** a model frozen on 1 January 2025 still beats persistence in 6 of 6 later quarters, and filings from companies never seen in training score ROC-AUC 0.890. A test on data from outside this dataset window runs automatically once newer quarter files are attached.
- **Model card (Cell 22):** intended use, out-of-scope uses, performance, failure modes and risks, generated from this run.

**Three rules that keep the result honest**
1. **Every number must beat the persistence rule** ("next quarter looks like this one"). The baseline table in Cell 7 is the most important output.
2. **Nothing from the label quarter may reach a feature.** Anything learned from data (clipping limits, medians, scalers, cluster centres) is fitted on the training period only.
3. **Evaluate on the future.** Train on the earliest label quarters (to Dec 2024), validate on the next (Jan to Sep 2025), test on the latest. Never shuffle. Cell 18 checks the same rule on the filing calendar. Choices are made on the validation period; the test period is reported, not used to choose (see "How the test period was used" at the end).

**Notebook map**
- **Part 1, S1 (Cells 1 to 13):** load, rebuild a clean standalone quarterly profit, build the panel, label and features, time split, baselines, models, lift, transition view and calibration, slices, importance, error review, rolling origin (12b), feature-group ablation (12c), fund-like filers (12d), save.
- **Part 2, next steps (Cells 14 to 17):** **U1** company archetypes by clustering, used as a feature, with an ablation; a **cheaper model** check; **S2**, a second target (positive equity today, negative equity next quarter) with the same machinery; one summary table.
- **Part 3, trust and use (Cells 18 to 22):** filing-date validation, confidence intervals, decision thresholds, later-period and unseen-company tests, model card.
- **At the end:** how to read the results, how the test period was used, limitations, notes for forks and edits, and the data-quality decisions behind the panel.

Needs the `sec-10k-10q-wide-v3` dataset (ETL v3.2) attached as input and scikit-learn (preinstalled on Kaggle). A full run takes about five minutes on CPU.

### Cell 1: imports and settings

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss, precision_recall_curve, adjusted_rand_score
from sklearn.model_selection import GroupKFold
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 170)

DATA_DIR = "/kaggle/input/datasets/siemmebrahtu/sec-10k-10q-wide-v3"   # same input as the EDA notebook (ETL v3.2)

# ---- S1 settings: change them here, everything below reads these ----
MIN_ASSETS = 1_000_000          # ratio features are left blank below this asset size (tiny shells make ratios explode)
MAX_DAYS_TO_FILE = 365          # filings later than this are catch-up filings and are excluded
GAP_RANGE = (75, 110)           # days between two consecutive quarter-ends (covers 13-week and 52/53-week calendars)
WINDOW_START = "2022-12-31"     # the first quarter-end the 14 quarter files can contain for a regular filer
TRAIN_END = "2024-12-31"        # label quarters up to here are TRAIN
VALID_END = "2025-09-30"        # label quarters after TRAIN_END and up to here are VALIDATION; later ones are TEST
SEED = 42

### Cell 2: load only the columns S1 needs (with a strict check by name)

In [ ]:
parquet_files = sorted(glob.glob(os.path.join(DATA_DIR, "20[0-9][0-9]q[1-4].parquet")))     # quarter files only, never superseded_filings.parquet
print(f"{len(parquet_files)} quarter files")
unified = pa.unify_schemas([pq.read_schema(p) for p in parquet_files], promote_options="permissive")

REQUIRED = ["adsh", "cik", "name", "sic_division", "form", "fp", "period", "dur_qtrs", "days_to_file", "is_spac", "is_shell", "is_amended",
            "has_negative_nonneg_tag", "NetIncomeLoss_qtr_src",
            "Assets", "AssetsCurrent", "Liabilities", "LiabilitiesCurrent", "EquityTotal", "RetainedEarningsAccumulatedDeficit",
            "CashAndCashEquivalentsAtCarryingValue",
            "NetIncomeLoss", "NetIncomeLoss_qtr", "NetIncomeLoss_ttm", "Revenues_qtr", "OperatingIncomeLoss_qtr",
            "NetCashProvidedByUsedInOperatingActivities_qtr", "NetCashProvidedByUsedInFinancingActivities_qtr"]
OPTIONAL = ["GrossProfit_qtr"]
missing = [c for c in REQUIRED if c not in unified.names]
if missing:
    raise KeyError(f"STOP: columns not in the dataset: {missing}. This is an old dataset version (run the ETL v3.2 notebook first) or DATA_DIR is wrong.")
cols = REQUIRED + [c for c in OPTIONAL if c in unified.names]
raw = ds.dataset(parquet_files, format="parquet", schema=unified).to_table(columns=cols).to_pandas()
raw["period"] = pd.to_datetime(raw["period"])
for c in ("is_spac", "is_shell", "is_amended", "has_negative_nonneg_tag"):
    raw[c] = raw[c].astype("boolean").astype("float")          # True/False/missing -> 1.0/0.0/NaN
print("loaded:", raw.shape)

### Cell 3: sort the panel and measure the gaps between quarters
Everything that follows relies on one idea: **a "previous" or "next" quarter only counts if its period end is about 3 months away** (75 to 110 days). Gaps (a missing filing, a catch-up filer) give a blank, never a wrong neighbour.

In [ ]:
# one filing per company per quarter (the latest one filed): a duplicate sits 0 days from its twin,
# so its neighbours would fall outside GAP_RANGE and get a blank or wrong previous/next quarter
raw = (raw.sort_values(["cik", "period", "days_to_file"])
                 .drop_duplicates(["cik", "period"], keep="last"))
d = raw.sort_values(["cik", "period"]).reset_index(drop=True)
g = d.groupby("cik")

gap_prev = (d["period"] - g["period"].shift(1)).dt.days
gap_next = (g["period"].shift(-1) - d["period"]).dt.days
prev_ok = gap_prev.between(*GAP_RANGE)
next_ok = gap_next.between(*GAP_RANGE)
print(f"rows: {len(d):,} | with a true previous quarter: {prev_ok.mean():.1%} | with a true next quarter: {next_ok.mean():.1%}")
print("duplicate (cik, period) pairs:", int(d.duplicated(["cik", "period"]).sum()))

### Cell 4: a clean standalone quarterly profit
The S1 label is the sign of next quarter's standalone profit, so this column has to be right.

In [ ]:
# Rebuild the standalone quarter from the CUMULATIVE profit by PERIOD ORDER: the previous row must end about one quarter
# earlier AND cover exactly one quarter fewer (dur_qtrs - 1). Chaining by fiscal-year label can take the previous row from
# the wrong fiscal year when the `fy` labels disagree (the NVIDIA example is in "Data-quality decisions" at the end).
prev_cum = g["NetIncomeLoss"].shift(1)
prev_dur = g["dur_qtrs"].shift(1)
chain_ok = prev_ok & (prev_dur == d["dur_qtrs"] - 1)
derived_chain = pd.Series(np.where(d["dur_qtrs"] == 1, d["NetIncomeLoss"], np.where(chain_ok, d["NetIncomeLoss"] - prev_cum, np.nan)), index=d.index)

is_reported = d["NetIncomeLoss_qtr_src"].eq("reported")          # a directly reported three-month fact: keep as is
d["ni_q"] = np.where(is_reported, d["NetIncomeLoss_qtr"], derived_chain)

# ---- how much did this matter? compare with the ETL's own derived values
etl_derived = d["NetIncomeLoss_qtr_src"].eq("derived")
cmp = d.loc[etl_derived, ["name", "form", "period", "dur_qtrs", "NetIncomeLoss_qtr"]].assign(chain=derived_chain[etl_derived])
both = cmp.dropna(subset=["NetIncomeLoss_qtr", "chain"])
tol = 0.01 * np.maximum(both["chain"].abs(), 1e4)
mismatch = (both["NetIncomeLoss_qtr"] - both["chain"]).abs() > tol
flips = (np.sign(both["NetIncomeLoss_qtr"]) != np.sign(both["chain"])) & mismatch
print(f"ETL-derived quarters: {int(etl_derived.sum()):,}")
print(f"  comparable with the period-chain value: {len(both):,}")
print(f"  DIFFER by more than 1%: {int(mismatch.sum()):,} ({mismatch.mean()*100:.2f}%), of which the SIGN is different: {int(flips.sum()):,}")
print(f"  ETL has a value but the period chain does not (no true previous quarter): {int((cmp['NetIncomeLoss_qtr'].notna() & cmp['chain'].isna()).sum()):,}")
print(f"  period chain has a value but the ETL does not: {int((cmp['NetIncomeLoss_qtr'].isna() & cmp['chain'].notna()).sum()):,}")
# the disagreement table is printed only when something disagrees
if mismatch.any():
    print("\nLargest disagreements (look these up by name):")
    print(both[mismatch].assign(diff=(both["NetIncomeLoss_qtr"] - both["chain"]).abs()).sort_values("diff", ascending=False).head(12)
      .rename(columns={"NetIncomeLoss_qtr": "etl_qtr", "chain": "period_chain_qtr"}).to_string(index=False))
print(f"\nni_q coverage: {d['ni_q'].notna().mean():.1%} (ETL NetIncomeLoss_qtr: {d['NetIncomeLoss_qtr'].notna().mean():.1%})")

# With ETL v3.2 the ETL's derived quarters are chained by period order too, so they must agree with the rebuild above
# (rounding cases at most). A disagreement rate above 0.5% means an older dataset version is attached.
rate = float(mismatch.mean()) if len(both) else 0.0
print("\n" + ("WARNING: " if rate > 0.005 else "OK: ") + f"{rate*100:.2f}% of ETL-derived quarters disagree with the period-chain value " +
      ("-> this looks like the OLD (v3.1) dataset. Run ETL v3.2, publish, and attach the new version." if rate > 0.005 else "-> the dataset is consistent with ETL v3.2."))

### Cell 5: the label, the lags and the features
Every feature is something the company had already reported **at this filing**. The label looks one quarter forward. Ratios use assets of at least `MIN_ASSETS` as the denominator.

In [ ]:
# ---------------- label (looks forward) ----------------
d["ni_q_next"] = g["ni_q"].shift(-1).where(next_ok)
d["next_period"] = g["period"].shift(-1).where(next_ok)
d["loss_now"] = (d["ni_q"] < 0).astype(float).where(d["ni_q"].notna())
d["loss_next"] = (d["ni_q_next"] < 0).astype(float).where(d["ni_q_next"].notna())

# ---------------- lags (look backwards, only through a true chain of consecutive quarters) ----------------
ok_g = prev_ok.groupby(d["cik"])
valid = {1: prev_ok}
valid[2] = valid[1] & ok_g.shift(1, fill_value=False)
valid[3] = valid[2] & ok_g.shift(2, fill_value=False)
valid[4] = valid[3] & ok_g.shift(3, fill_value=False)
for k in range(1, 5):
    d[f"lag_ni_{k}"] = g["ni_q"].shift(k).where(valid[k])
    d[f"lag_rev_{k}"] = g["Revenues_qtr"].shift(k).where(valid[k])

# ---------------- features ----------------
A = d["Assets"].where(d["Assets"] >= MIN_ASSETS)
def ratio(num, den): return num / den.where(den > 0)
def non_neg(x): return x.where(x >= 0)
def signed_log(x): return np.sign(x) * np.log10(1 + np.abs(x))

F = pd.DataFrame(index=d.index)
# profit history, scaled by assets
F["ni_a0"] = d["ni_q"] / A
for k in range(1, 5):
    F[f"ni_a{k}"] = d[f"lag_ni_{k}"] / A
F["loss_now"] = d["loss_now"]
for k in (1, 2, 3):
    F[f"loss_lag{k}"] = (d[f"lag_ni_{k}"] < 0).astype(float).where(d[f"lag_ni_{k}"].notna())
alive = (F["loss_now"] == 1)
streak = alive.astype(int)
for k in (1, 2, 3):
    alive = alive & (F[f"loss_lag{k}"] == 1)
    streak = streak + alive.astype(int)
F["loss_streak"] = streak.where(F["loss_now"].notna())           # consecutive loss quarters up to now (a gap or a profit ends the streak)
F["roa_ttm"] = d["NetIncomeLoss_ttm"] / A
# margins and growth
F["net_margin_q"] = ratio(d["ni_q"], d["Revenues_qtr"])
F["op_margin_q"] = ratio(d["OperatingIncomeLoss_qtr"], d["Revenues_qtr"])
F["rev_qoq"] = d["Revenues_qtr"] / d["lag_rev_1"].where(d["lag_rev_1"] > 0) - 1
F["rev_yoy"] = d["Revenues_qtr"] / d["lag_rev_4"].where(d["lag_rev_4"] > 0) - 1
# cash quality
F["ocf_a"] = d["NetCashProvidedByUsedInOperatingActivities_qtr"] / A
F["accrual_a"] = (d["ni_q"] - d["NetCashProvidedByUsedInOperatingActivities_qtr"]) / A
F["financing_a"] = d["NetCashProvidedByUsedInFinancingActivities_qtr"] / A
# leverage, liquidity, capital
F["debt_to_assets"] = ratio(non_neg(d["Liabilities"]), A)
F["current_ratio"] = ratio(non_neg(d["AssetsCurrent"]), d["LiabilitiesCurrent"])
F["cash_to_cur_liab"] = ratio(non_neg(d["CashAndCashEquivalentsAtCarryingValue"]), d["LiabilitiesCurrent"])
F["equity_ratio"] = d["EquityTotal"] / A
F["retained_a"] = d["RetainedEarningsAccumulatedDeficit"] / A
# size
F["log_assets"] = np.log10(d["Assets"].where(d["Assets"] > 0))
F["log_rev_q"] = signed_log(d["Revenues_qtr"])
# filing behaviour and flags (all known at filing time)
F["days_to_file"] = d["days_to_file"].clip(0, MAX_DAYS_TO_FILE)
F["is_spac"] = d["is_spac"]; F["is_shell"] = d["is_shell"]      # the amendment flag is left out: whether a filing gets amended is only known later
# calendar and sector (categorical)
F["next_fq"] = d["fp"].map({"Q1": 2, "Q2": 3, "Q3": 4, "FY": 1}).astype(float)      # which fiscal quarter the label quarter is
F["div_code"] = pd.Series(pd.Categorical(d["sic_division"]).codes, index=d.index).astype(float).where(lambda s: s >= 0)

# EPS is deliberately NOT a feature: the raw per-share values contain scale errors (for example -713,794 and +1.85 billion per share)
# that cannot be rebuilt from other columns, and profit scaled by assets carries the same information.

CAT_COLS = ["next_fq", "div_code"]
FLAG_COLS = ["loss_now", "loss_lag1", "loss_lag2", "loss_lag3", "loss_streak", "is_spac", "is_shell"]
FEATURES = list(F.columns)
CLIP_COLS = [c for c in FEATURES if c not in CAT_COLS + FLAG_COLS]
print(f"{len(FEATURES)} features: {len(CLIP_COLS)} continuous (clipped), {len(FLAG_COLS)} flags, {len(CAT_COLS)} categorical")
print("\nshare of rows with a value, per feature:")
print((F.notna().mean() * 100).round(1).sort_values().to_string())

### Cell 6: choose the rows, then split by time
The split is on the period of the **label quarter**, so a training row's answer can never come from the future of the test period.

In [ ]:
D = pd.concat([d, F.add_prefix("f_")], axis=1)

# A row is kept only with a standalone profit now and a true next quarter (the label), a period inside the window, a plausible filing
# delay (a negative delay is impossible, a very late one is a catch-up filing) and no negative value in a tag that can never be negative.
funnel = []
m = pd.Series(True, index=D.index); funnel.append(("all filings", int(m.sum())))
m &= D["loss_now"].notna();                          funnel.append(("+ a standalone profit this quarter", int(m.sum())))
m &= D["loss_next"].notna();                         funnel.append(("+ a true next quarter with a profit figure (the label)", int(m.sum())))
m &= D["period"] >= pd.Timestamp(WINDOW_START);      funnel.append((f"+ period on or after {WINDOW_START}", int(m.sum())))
m &= D["days_to_file"].between(0, MAX_DAYS_TO_FILE); funnel.append((f"+ days_to_file between 0 and {MAX_DAYS_TO_FILE}", int(m.sum())))
m &= D["has_negative_nonneg_tag"].fillna(0) == 0;    funnel.append(("+ no negative value in a never-negative tag", int(m.sum())))
print(pd.DataFrame(funnel, columns=["step", "rows left"]).to_string(index=False))
print("\nnegative days_to_file rows:", int((D["days_to_file"] < 0).sum()), "| more than 365:", int((D["days_to_file"] > MAX_DAYS_TO_FILE).sum()))

S = D[m].copy()
S["split"] = np.where(S["next_period"] <= pd.Timestamp(TRAIN_END), "train",
                      np.where(S["next_period"] <= pd.Timestamp(VALID_END), "valid", "test"))
summary = S.groupby("split").agg(rows=("adsh", "size"), companies=("cik", "nunique"), loss_rate=("loss_next", "mean"),
                                 first_label_quarter=("next_period", "min"), last_label_quarter=("next_period", "max")).loc[["train", "valid", "test"]]
print("\n", summary)
print("\nShare of profitable-now rows (the transition population) in each split:",
      (S.groupby("split")["loss_now"].apply(lambda s: (s == 0).mean()) * 100).round(1).to_dict())
tr, va, te = (S[S["split"] == s] for s in ("train", "valid", "test"))
assert len(tr) > 0 and len(va) > 0 and len(te) > 0, "an empty split: change TRAIN_END / VALID_END"
assert tr["next_period"].max() < va["next_period"].min() <= va["next_period"].max() < te["next_period"].min()

### Cell 7: the baselines (the numbers every model must beat)
Three rules that need no model: the majority rate, **persistence** (next quarter's loss = this quarter's loss) and a **history table** (the training-set loss rate for each pattern of the last two quarters).

In [ ]:
def metrics(y, p):
    y = np.asarray(y); p = np.asarray(p, dtype=float)
    return dict(n=len(y), loss_rate=y.mean(), roc_auc=roc_auc_score(y, p), pr_auc=average_precision_score(y, p),
                brier=brier_score_loss(y, p), log_loss=log_loss(y, np.clip(p, 1e-6, 1 - 1e-6)))

# ---- majority, persistence
base_rate = tr["loss_next"].mean()
# ---- history table learned on TRAIN only: P(loss next | loss now, loss last quarter)
key = lambda df: list(zip(df["f_loss_now"], df["f_loss_lag1"].fillna(-1)))
hist = tr.assign(k=key(tr)).groupby("k")["loss_next"].agg(["mean", "size"])
now_only = tr.groupby("f_loss_now")["loss_next"].mean()
def history_prob(df):
    out = []
    for (now, lag), now_ in zip(key(df), df["f_loss_now"]):
        if (now, lag) in hist.index and hist.loc[[(now, lag)], "size"].iloc[0] >= 50:
            out.append(hist.loc[[(now, lag)], "mean"].iloc[0])
        else:
            out.append(now_only[now_])
    return np.array(out)

rows = []
for name, df in (("valid", va), ("test", te)):
    y = df["loss_next"].values
    rows.append(dict(split=name, model="majority (train loss rate)", **metrics(y, np.full(len(y), base_rate))))
    rows.append(dict(split=name, model="persistence (next = now)", **metrics(y, df["f_loss_now"].values)))
    rows.append(dict(split=name, model="history table (last 2 quarters)", **metrics(y, history_prob(df))))
baseline_table = pd.DataFrame(rows)
print(baseline_table.round(4).to_string(index=False))

print("\nHow persistent is a loss? P(loss next quarter), by this quarter's result and by split:")
tt = S.assign(now=np.where(S["f_loss_now"] == 1, "loss now", "profit now"))
print(tt.pivot_table(index="now", columns="split", values="loss_next", aggfunc="mean")[["train", "valid", "test"]].round(4))
print("rows:")
print(tt.pivot_table(index="now", columns="split", values="loss_next", aggfunc="size")[["train", "valid", "test"]])

### Cell 8: the models
A regularised logistic regression (transparent) and gradient boosting (handles blanks and interactions). Both see exactly the same features and the same training rows. In the cross-validation, the clip limits, medians and scaler are refitted inside every fold, so held-out companies never shape the preparation of their own rows.

In [ ]:
# Ratio features have huge tails, so they are capped at the 1st and 99th percentile. The limits come from the TRAINING rows only:
# limits computed on all rows would pass information from the validation and test periods into training.
Xcols = ["f_" + c for c in FEATURES]
lo = tr[["f_" + c for c in CLIP_COLS]].quantile(0.01)
hi = tr[["f_" + c for c in CLIP_COLS]].quantile(0.99)
def prep(df):
    X = df[Xcols].copy()
    cc = ["f_" + c for c in CLIP_COLS]
    X[cc] = X[cc].clip(lower=lo, upper=hi, axis=1)
    X.columns = FEATURES
    return X
Xtr, Xva, Xte = prep(tr), prep(va), prep(te)
ytr, yva, yte = tr["loss_next"].values, va["loss_next"].values, te["loss_next"].values

num_cols = [c for c in FEATURES if c not in CAT_COLS]
logit = Pipeline([
    ("prep", ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)), ("sc", StandardScaler())]), num_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value=-1)), ("oh", OneHotEncoder(handle_unknown="ignore"))]), CAT_COLS),
    ])),
    ("clf", LogisticRegression(C=0.5, max_iter=1000)),
])
cat_idx = [FEATURES.index(c) for c in CAT_COLS]
gbm = HistGradientBoostingClassifier(learning_rate=0.06, max_iter=400, max_leaf_nodes=31, min_samples_leaf=40, l2_regularization=1.0,
                                     categorical_features=cat_idx, early_stopping=True, n_iter_no_change=20, validation_fraction=0.1, random_state=SEED)

# ---- company-aware cross-validation inside TRAIN (no company on both sides of a fold)
# Every learned quantity is refitted inside each fold: the clip limits here, the medians and scaler inside the logistic pipeline.
# The held-out companies of a fold therefore contribute nothing to the preparation of the rows they are scored on.
clip_x = ["f_" + c for c in CLIP_COLS]
def prep_fold(fit_df, df):
    X = df[Xcols].copy()
    X[clip_x] = X[clip_x].clip(lower=fit_df[clip_x].quantile(0.01), upper=fit_df[clip_x].quantile(0.99), axis=1)
    X.columns = FEATURES
    return X
gkf = GroupKFold(n_splits=5)
for name, model in (("logistic", logit), ("gradient boosting", gbm)):
    aucs, prs = [], []
    for a, b in gkf.split(Xtr, ytr, groups=tr["cik"].values):
        fit_df, held_df = tr.iloc[a], tr.iloc[b]
        model.fit(prep_fold(fit_df, fit_df), ytr[a]); p = model.predict_proba(prep_fold(fit_df, held_df))[:, 1]
        aucs.append(roc_auc_score(ytr[b], p)); prs.append(average_precision_score(ytr[b], p))
    print(f"{name:18s} GroupKFold by company (5 folds): ROC-AUC {np.mean(aucs):.3f} +/- {np.std(aucs):.3f} | PR-AUC {np.mean(prs):.3f} +/- {np.std(prs):.3f}")

# ---- final fit on all TRAIN rows, evaluate on the FUTURE
logit.fit(Xtr, ytr); gbm.fit(Xtr, ytr)
pred = {}
for split, X, y in (("valid", Xva, yva), ("test", Xte, yte)):
    pred[("logistic", split)] = logit.predict_proba(X)[:, 1]
    pred[("gradient boosting", split)] = gbm.predict_proba(X)[:, 1]
model_rows = [dict(split=s, model=m, **metrics(y, pred[(m, s)])) for s, y in (("valid", yva), ("test", yte)) for m in ("logistic", "gradient boosting")]
results = pd.concat([baseline_table, pd.DataFrame(model_rows)], ignore_index=True).sort_values(["split", "roc_auc"], ascending=[True, False])
print("\n", results.round(4).to_string(index=False))
print(f"\nGradient boosting stopped after {gbm.n_iter_} iterations.")

### Cell 9: does it add anything beyond persistence? Lift, transition view, calibration
The key comparison. **Lift** is the model's ROC-AUC and PR-AUC minus the persistence rule's, on the same rows.

In [ ]:
def get(split, model, col):
    return results[(results["split"] == split) & (results["model"] == model)][col].iloc[0]

print("LIFT over persistence (positive = the model adds something):")
lift = pd.DataFrame([{"split": s, "model": m,
                      "roc_auc_lift": get(s, m, "roc_auc") - get(s, "persistence (next = now)", "roc_auc"),
                      "pr_auc_lift": get(s, m, "pr_auc") - get(s, "persistence (next = now)", "pr_auc"),
                      "brier_gain": get(s, "persistence (next = now)", "brier") - get(s, m, "brier")}
                     for s in ("valid", "test") for m in ("history table (last 2 quarters)", "logistic", "gradient boosting")])
print(lift.round(4).to_string(index=False))

print("\nTRANSITION view on the TEST period (only rows that are profitable NOW: who turns to a loss?):")
trans_rows = []
for model in ("history table (last 2 quarters)", "logistic", "gradient boosting"):
    p = history_prob(te) if model.startswith("history") else pred[(model, "test")]
    sel = (te["f_loss_now"] == 0).values
    if sel.sum() > 50 and len(np.unique(yte[sel])) == 2:
        trans_rows.append(dict(model=model, rows=int(sel.sum()), turn_to_loss_rate=yte[sel].mean(), roc_auc=roc_auc_score(yte[sel], p[sel]), pr_auc=average_precision_score(yte[sel], p[sel])))
print(pd.DataFrame(trans_rows).round(4).to_string(index=False))
print("(persistence has no ranking power here: every row is 'profit now', so its score is the same for all of them)")

print("\nCALIBRATION of gradient boosting on TEST (does a predicted 30% happen about 30% of the time?):")
cal = pd.DataFrame({"p": pred[("gradient boosting", "test")], "y": yte})
cal["bucket"] = pd.qcut(cal["p"], 10, duplicates="drop")
cal_t = cal.groupby("bucket").agg(mean_predicted=("p", "mean"), actual_loss_rate=("y", "mean"), rows=("y", "size")).round(3)
cal_t.index = [f"{iv.left:.3f} to {iv.right:.3f}" for iv in cal_t.index]      # readable bucket labels
print(cal_t.to_string())
fig, ax = plt.subplots(figsize=(4.6, 4.4))
ax.plot([0, 1], [0, 1], "--", color="grey"); ax.plot(cal_t["mean_predicted"], cal_t["actual_loss_rate"], "o-", color="#0F766E")
ax.set_xlabel("mean predicted probability"); ax.set_ylabel("actual loss rate"); ax.set_title("Calibration (test)")
plt.tight_layout(); plt.show()

### Cell 10: where does it work and where does it not? (slices)
A single score hides big differences. Slices below need at least 300 rows and both outcomes.

In [ ]:
def slice_table(df, p, by, min_rows=300):
    out = []
    for key_, g_ in df.groupby(by):
        idx = g_.index
        yy = g_["loss_next"].values
        if len(g_) >= min_rows and len(np.unique(yy)) == 2:
            pp = pd.Series(p, index=df.index).loc[idx].values
            out.append({"slice": key_, "rows": len(g_), "loss_rate": yy.mean(), "roc_auc": roc_auc_score(yy, pp),
                        "persistence_roc_auc": roc_auc_score(yy, g_["f_loss_now"].values) if g_["f_loss_now"].nunique() == 2 else np.nan})
    return pd.DataFrame(out).round(3)

pt = pd.Series(pred[("gradient boosting", "test")], index=te.index)
te_ = te.copy()
te_["size_band"] = pd.qcut(te_["Assets"], 3, labels=["small", "middle", "large"], duplicates="drop")
te_["segment"] = np.where(te_["f_is_spac"] == 1, "SPAC", np.where(te_["f_is_shell"] == 1, "shell", "operating company"))
for by in ("sic_division", "segment", "size_band", "fp"):
    print(f"\nTEST ROC-AUC of gradient boosting by {by} (versus persistence on the same rows):")
    print(slice_table(te_, pt.values, by).to_string(index=False))

### Cell 11: what drives the predictions? (permutation importance on the validation period)
Importance = how much the validation ROC-AUC drops when one feature is shuffled. Correlated features share credit, so read it as a ranking, not as causes.

In [ ]:
imp = permutation_importance(gbm, Xva, yva, scoring="roc_auc", n_repeats=5, random_state=SEED, n_jobs=1)
imp_t = pd.DataFrame({"feature": FEATURES, "auc_drop": imp.importances_mean, "std": imp.importances_std}).sort_values("auc_drop", ascending=False)
print(imp_t.head(15).round(4).to_string(index=False))
fig, ax = plt.subplots(figsize=(7, 5))
top = imp_t.head(15).iloc[::-1]
ax.barh(top["feature"], top["auc_drop"], xerr=top["std"], color="#B45309"); ax.set_xlabel("drop in validation ROC-AUC when shuffled")
ax.set_title("Gradient boosting: permutation importance"); plt.tight_layout(); plt.show()

### Cell 12: the biggest mistakes, by company name
A model is not finished until its worst errors can be explained. The list below shows the most confident wrong predictions on the test period, with the company names, so each one can be checked against the filing: a data problem (a wrong quarter, a scale error) or a real surprise?

In [ ]:
err = te[["name", "form", "period", "next_period", "ni_q", "ni_q_next", "f_loss_now", "loss_next"]].copy()
err["p_loss"] = pred[("gradient boosting", "test")]
err["error"] = (err["p_loss"] - err["loss_next"]).abs()
print("Most confident WRONG predictions on the test period:")
print(err.sort_values("error", ascending=False).head(15).rename(columns={"f_loss_now": "loss_now"}).round(3).to_string(index=False))

### Cell 12b: rolling-origin evaluation
For each later calendar quarter, train on all earlier label quarters and test on that one quarter. If the lift over persistence stays positive in every quarter, the result does not depend on one lucky test period.

In [ ]:
ROLLING_FIRST_TEST = "2024Q3"; ROLLING_MIN_TRAIN = 5000; ROLLING_MIN_TEST = 300

# fit_predict: the one gradient-boosting recipe used from here on (rolling origin, ablations, U1, S2 and Part 3), so every model follows
# the same clipping rule and settings. Any target and categorical columns can be passed; the defaults give the S1 model.
def fit_predict(train_df, eval_dfs, features=FEATURES, max_iter=250, target="loss_next", cat_cols=None, return_model=False):
    """Gradient boosting with the same rules as Cell 8: clip limits from the TRAINING rows only."""
    features = list(features)
    cat_cols = CAT_COLS if cat_cols is None else cat_cols
    clip_cols = ["f_" + c for c in CLIP_COLS if c in features]
    lo_ = train_df[clip_cols].quantile(0.01); hi_ = train_df[clip_cols].quantile(0.99)
    def prep_(df):
        X = df[["f_" + c for c in features]].copy()
        X[clip_cols] = X[clip_cols].clip(lower=lo_, upper=hi_, axis=1)
        X.columns = features
        return X
    cats = [features.index(c) for c in cat_cols if c in features] or None
    model = HistGradientBoostingClassifier(learning_rate=0.08, max_iter=max_iter, max_leaf_nodes=31, min_samples_leaf=40, l2_regularization=1.0,
                                           categorical_features=cats, early_stopping=True, n_iter_no_change=15, validation_fraction=0.1, random_state=SEED)
    model.fit(prep_(train_df), train_df[target].values)
    preds = [model.predict_proba(prep_(df))[:, 1] for df in eval_dfs]
    return (preds, model, prep_) if return_model else preds

S["label_q"] = S["next_period"].dt.to_period("Q")
roll = []
for q in sorted(S["label_q"].unique()):
    if q < pd.Period(ROLLING_FIRST_TEST):
        continue
    trn, tst = S[S["label_q"] < q], S[S["label_q"] == q]
    y = tst["loss_next"].values
    if len(trn) < ROLLING_MIN_TRAIN or len(tst) < ROLLING_MIN_TEST or len(np.unique(y)) < 2:
        continue
    p, = fit_predict(trn, [tst])
    pers = tst["f_loss_now"].values
    roll.append(dict(label_quarter=str(q), train_rows=len(trn), test_rows=len(tst), loss_rate=y.mean(),
                     gbm_roc_auc=roc_auc_score(y, p), persistence_roc_auc=roc_auc_score(y, pers),
                     gbm_pr_auc=average_precision_score(y, p), persistence_pr_auc=average_precision_score(y, pers)))
roll = pd.DataFrame(roll)
roll["roc_auc_lift"] = roll["gbm_roc_auc"] - roll["persistence_roc_auc"]
roll["pr_auc_lift"] = roll["gbm_pr_auc"] - roll["persistence_pr_auc"]
print(roll.round(3).to_string(index=False))
print(f"\nLift over persistence: ROC-AUC mean {roll['roc_auc_lift'].mean():.3f} (min {roll['roc_auc_lift'].min():.3f}), PR-AUC mean {roll['pr_auc_lift'].mean():.3f} (min {roll['pr_auc_lift'].min():.3f})")
print("Quarters where the model did NOT beat persistence on ROC-AUC:", int((roll["roc_auc_lift"] <= 0).sum()), "of", len(roll))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(roll["label_quarter"], roll["gbm_roc_auc"], "o-", color="#0F766E", label="gradient boosting")
ax.plot(roll["label_quarter"], roll["persistence_roc_auc"], "s--", color="#94A3B8", label="persistence")
ax.set_ylabel("ROC-AUC on that quarter"); ax.set_title("Rolling origin: each quarter tested on a model trained only on earlier quarters")
ax.legend(); plt.xticks(rotation=45); plt.tight_layout(); plt.show()

### Cell 12c: which feature groups carry the result? (ablation)
Same model, same train and test rows; only the feature list changes. **Cumulative** adds one group at a time; **leave one out** removes one group from the full set. A group whose removal costs little is not pulling its weight. Read the **validation** columns to decide anything; the test columns are shown for transparency and changed no choice in this notebook.

In [ ]:
GROUPS = {
    "history":       ["loss_now", "loss_lag1", "loss_lag2", "loss_lag3", "loss_streak"],
    "profit_scale":  ["ni_a0", "ni_a1", "ni_a2", "ni_a3", "ni_a4", "roa_ttm", "net_margin_q", "op_margin_q", "rev_qoq", "rev_yoy"],
    "cash_quality":  ["ocf_a", "accrual_a", "financing_a"],
    "balance_sheet": ["debt_to_assets", "current_ratio", "cash_to_cur_liab", "equity_ratio", "retained_a"],
    "size_context":  ["log_assets", "log_rev_q", "days_to_file", "is_spac", "is_shell", "next_fq", "div_code"],
}
assert set(sum(GROUPS.values(), [])) == set(FEATURES), "every feature must sit in exactly one group"

def score(features, label):
    pv, pt = fit_predict(tr, [va, te], features=features)
    return dict(feature_set=label, n_features=len(features),
                valid_roc_auc=roc_auc_score(yva, pv), valid_pr_auc=average_precision_score(yva, pv),
                test_roc_auc=roc_auc_score(yte, pt), test_pr_auc=average_precision_score(yte, pt))

rows, acc = [], []
for name, feats in GROUPS.items():
    acc += feats
    rows.append(score(list(acc), "cumulative: + " + name))
all_feats = list(FEATURES)
rows.append(score(all_feats, "ALL features"))
for name, feats in GROUPS.items():
    rows.append(score([f for f in all_feats if f not in feats], "all minus " + name))
rows.append(score([f for f in all_feats if f != "ni_a3"], "all minus ni_a3 only (seasonal analogue)"))
abl = pd.DataFrame(rows)
full = abl[abl["feature_set"] == "ALL features"].iloc[0]
abl["test_roc_auc_vs_all"] = abl["test_roc_auc"] - full["test_roc_auc"]
print(abl.round(4).to_string(index=False))
print("\nReference: persistence test ROC-AUC", round(roc_auc_score(yte, te["f_loss_now"].values), 4), "| history table test ROC-AUC",
      round(float(results[(results["split"] == "test") & (results["model"] == "history table (last 2 quarters)")]["roc_auc"].iloc[0]), 4))

### Cell 12d: fund-like filers
Private-credit funds and BDC-style filers report a "net increase in net assets" that moves with market valuations, so many of them swing the same way in the same quarter. This is a **diagnostic on company names** (a rough pattern, not a feature, and it will mislabel some companies).

In [ ]:
# a narrow pattern for private-credit funds and BDCs, plus an exclusion list for names that only look similar
# (captive finance arms, banks, REITs, royalty trusts, SPACs, tax-credit funds)
FUND_PATTERN = r"PRIVATE CREDIT|DIRECT LENDING|SPECIALTY LENDING|LENDING CORP|LENDING FUND|LENDING PARTNERS|\bBDC\b|CAPITAL TRUST|CREDIT FUND|CREDIT INCOME|CREDIT OPPORTUNITIES|CREDIT STRATEGIES|INCOME CORP|SENIOR LOAN"
FUND_EXCLUDE = r"REAL ESTATE|REALTY|ROYALTY|MOTOR CREDIT|NORTHERN TRUST|ACQUISITION|TAX CREDIT"
names_up = S["name"].str.upper()
S["fund_like"] = (names_up.str.contains(FUND_PATTERN, regex=True, na=False)
                                    & ~names_up.str.contains(FUND_EXCLUDE, regex=True, na=False))
print(f"fund-like names: {int(S['fund_like'].sum()):,} of {len(S):,} rows ({S['fund_like'].mean()*100:.1f}%), {S.loc[S['fund_like'], 'cik'].nunique():,} companies")
print("\nExamples:", ", ".join(S.loc[S["fund_like"], "name"].drop_duplicates().head(8).tolist()))

print("\nLoss rate of the NEXT quarter by label quarter (a quarter where the fund-like rate jumps while the others do not is a common shock the model cannot see):")
lr = S.groupby(["label_q", "fund_like"])["loss_next"].agg(["mean", "size"]).unstack()
lr.columns = [f"{'fund-like' if b else 'other'}_{a}" for a, b in lr.columns]
print(lr.reindex(columns=["other_mean", "fund-like_mean", "other_size", "fund-like_size"]).round(3).to_string())

tt = te.copy(); tt["fund_like"] = S.loc[te.index, "fund_like"]; tt["p"] = pred[("gradient boosting", "test")]
print("\nTEST ROC-AUC of gradient boosting, fund-like versus other:")
res_ = []
for flag, g_ in tt.groupby("fund_like"):
    yy = g_["loss_next"].values
    if len(g_) >= 100 and len(np.unique(yy)) == 2:
        res_.append({"group": "fund-like" if flag else "other", "rows": len(g_), "loss_rate": yy.mean(), "roc_auc": roc_auc_score(yy, g_["p"].values),
                     "persistence_roc_auc": roc_auc_score(yy, g_["f_loss_now"].values) if g_["f_loss_now"].nunique() == 2 else np.nan})
print(pd.DataFrame(res_).round(3).to_string(index=False))

### Cell 13: save the S1 results

In [ ]:
out = te[["adsh", "cik", "name", "period", "next_period", "loss_next"]].copy()
out["p_logistic"] = pred[("logistic", "test")]; out["p_gbm"] = pred[("gradient boosting", "test")]; out["persistence"] = te["f_loss_now"].values
out.to_parquet("/kaggle/working/s1_test_predictions.parquet", index=False)
results.round(5).to_csv("/kaggle/working/s1_metrics.csv", index=False)
print("saved", out.shape, "-> /kaggle/working/s1_test_predictions.parquet and s1_metrics.csv")

---
# Part 2: the next steps

Part 1 answered the main question. Part 2 takes the three follow-ups it raised, each with the same honesty rules (train on the past only, beat a simple rule, test on the future):

| Cell | Step | Question |
|---|---|---|
| 14 | **U1: company archetypes** | Do companies fall into a few recognisable types, and does knowing the type improve the S1 forecast? |
| 15 | **A cheaper model** | How much of the lift does a small, transparent model keep? |
| 16a-16c | **S2: equity turns negative** | Same machinery, second target: which companies with positive equity today will report negative equity next quarter? |
| 17 | **Save and summarise** | One table with every headline number, plus the files for the Output tab |

### Cell 14: U1, company archetypes (clustering) as a feature, plus an ablation
**What:** k-means groups the training rows into a handful of *archetypes* using 12 balance-sheet, profitability, cash-flow and size features (no label is used). Every filing then gets the archetype it is closest to.

**Rule 2 still holds:** the clipping limits, the medians, the scaler and the cluster centres are all fitted on the TRAIN rows only; validation and test filings are only *assigned* to the existing centres.

**Why seven archetypes, when the silhouette keeps rising up to ten?** The number is not tuned by hand. The rule is: take the highest silhouette among the solutions in which **every archetype holds at least 2% of the training rows**. The silhouette (how far apart the clusters are) rises with almost every extra cluster on data with heavy tails, because each new cluster can split off a small group of extreme values. Those small groups raise the average separation without adding a new, recognisable type of company, and a cluster of a few hundred filings is too small to describe or to trust over time. The cell prints the evidence for each k: the size of the smallest cluster, how many clusters fall under 2%, how **stable** the solution is (two fits on different halves of the training rows, compared with the adjusted Rand index) and how **useful** it is (how well the archetype alone ranks validation rows). The explanation under the table is written from those numbers.

**How to read it:** the profile table describes each archetype in plain terms. The ablation then asks whether the archetype adds anything on top of the features the model already has; the verdict is taken from the validation period, and the test period only confirms it. A clustering that describes companies well but adds little to the forecast is still a useful *description*. It is just not a useful *feature*.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# ---- U1 settings
ARCH_FEATURES = ["ni_a0", "roa_ttm", "ocf_a", "accrual_a", "financing_a", "debt_to_assets", "equity_ratio",
                 "retained_a", "current_ratio", "cash_to_cur_liab", "log_assets", "log_rev_q"]
ARCH_K_RANGE = range(4, 11)     # numbers of archetypes to try
ARCH_K = None                   # set an integer here to skip the automatic choice
ARCH_SAMPLE = 8000              # rows used to score each k (the silhouette score is slow on all rows)

# ---- the clustering matrix: same TRAIN-only clip limits as Cell 8, then TRAIN-only medians and scaling
lo_bare, hi_bare = lo.rename(lambda s: s[2:]), hi.rename(lambda s: s[2:])     # Cell 8's limits without the "f_" prefix
def arch_matrix(df):
    X = df[["f_" + c for c in ARCH_FEATURES]].copy()
    X.columns = ARCH_FEATURES
    cc = [c for c in ARCH_FEATURES if c in CLIP_COLS]
    X[cc] = X[cc].clip(lower=lo_bare[cc], upper=hi_bare[cc], axis=1)
    X["log_rev_q"] = X["log_rev_q"].fillna(0.0)       # no revenue reported = a pre-revenue company, itself a type worth keeping
    return X

arch_prep = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]).fit(arch_matrix(tr))
Ztr = arch_prep.transform(arch_matrix(tr))
samp = np.random.RandomState(SEED).choice(len(Ztr), size=min(ARCH_SAMPLE, len(Ztr)), replace=False)

# ---- how many archetypes? Five checks per k, all on TRAIN rows (the validation column only READS validation rows):
#   silhouette              how well separated the clusters are (higher = better)
#   smallest_cluster_share  the size of the smallest cluster (a cluster of a few hundred rows is usually a group of extreme values)
#   loss_rate_spread        how different the clusters' training loss rates are
#   stability_ari           do two k-means fits on different random halves of TRAIN agree? (adjusted Rand index: 1 = identical, 0 = chance)
#   valid_roc_auc_archetype_only   how well "the training loss rate of the row's cluster" ranks VALIDATION rows (usefulness, no model)
ARCH_MIN_SHARE = 0.02           # every archetype must hold at least 2% of the training rows
ARCH_STABILITY_REPEATS = 3      # random half-splits per k for the stability check
if ARCH_K is None:
    sil = []
    Zva = arch_prep.transform(arch_matrix(va))
    rs = np.random.RandomState(SEED)
    for k in ARCH_K_RANGE:
        km_k = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(Ztr)
        rates = pd.Series(ytr).groupby(km_k.labels_).mean()
        counts = np.bincount(km_k.labels_, minlength=k)
        aris = []
        for r in range(ARCH_STABILITY_REPEATS):
            perm = rs.permutation(len(Ztr)); half_a, half_b = perm[: len(Ztr) // 2], perm[len(Ztr) // 2:]
            lab_a = KMeans(n_clusters=k, n_init=4, random_state=SEED + r).fit(Ztr[half_a]).predict(Ztr)
            lab_b = KMeans(n_clusters=k, n_init=4, random_state=SEED + 100 + r).fit(Ztr[half_b]).predict(Ztr)
            aris.append(adjusted_rand_score(lab_a, lab_b))
        sil.append({"k": k, "silhouette": silhouette_score(Ztr[samp], km_k.labels_[samp]),
                    "smallest_cluster_share": counts.min() / len(Ztr), "smallest_cluster_rows": int(counts.min()),
                    "clusters_under_2pct": int((counts / len(Ztr) < ARCH_MIN_SHARE).sum()),
                    "loss_rate_spread": rates.max() - rates.min(), "stability_ari": float(np.mean(aris)),
                    "valid_roc_auc_archetype_only": roc_auc_score(yva, rates.reindex(km_k.predict(Zva)).values)})
    sil = pd.DataFrame(sil)
    print("Choosing the number of archetypes:")
    print(sil.round(3).to_string(index=False))
    ok_k = sil[sil["smallest_cluster_share"] >= ARCH_MIN_SHARE]            # solutions where every archetype is a real group of companies
    ARCH_K = int((ok_k if len(ok_k) else sil).sort_values("silhouette", ascending=False)["k"].iloc[0])

    # ---- why this k? The silhouette keeps rising with k, so show where the extra separation comes from
    ch, best = sil.set_index("k").loc[ARCH_K], sil.loc[sil["silhouette"].idxmax()]
    bigger = sil[sil["k"] > ARCH_K]
    if len(ok_k):
        print(f"\nWhy {ARCH_K}: it has the highest silhouette among the solutions in which every archetype holds at least {ARCH_MIN_SHARE:.0%} of the training rows.")
    else:
        print(f"\nWhy {ARCH_K}: no solution keeps every archetype above {ARCH_MIN_SHARE:.0%} of the rows, so the highest silhouette is used.")
    if int(best["k"]) > ARCH_K:
        print(f"- Above k={ARCH_K} the silhouette still rises ({ch['silhouette']:.3f} -> {best['silhouette']:.3f} at k={int(best['k'])}), but "
              f"{int((bigger['clusters_under_2pct'] > 0).sum())} of those {len(bigger)} solutions contain a cluster under {ARCH_MIN_SHARE:.0%} "
              f"(at k={int(best['k'])} the smallest has {int(best['smallest_cluster_rows']):,} rows, {best['smallest_cluster_share']:.1%}). "
              "With heavy-tailed ratios, extra clusters mostly split off small groups of extreme values: that raises the average separation "
              "without adding a new mainstream type of company.")
    else:
        print(f"- No larger k has a higher silhouette than k={ARCH_K} ({ch['silhouette']:.3f}).")
    print(f"- Stability: two fits on different halves of the training rows agree with ARI {ch['stability_ari']:.2f} at k={ARCH_K} "
          f"(range over k: {sil['stability_ari'].min():.2f} to {sil['stability_ari'].max():.2f}).")
    print(f"- Usefulness: the archetype alone ranks validation rows with ROC-AUC {ch['valid_roc_auc_archetype_only']:.3f} at k={ARCH_K} "
          f"(range over k: {sil['valid_roc_auc_archetype_only'].min():.3f} to {sil['valid_roc_auc_archetype_only'].max():.3f}), "
          + ("so the choice of k barely changes what the archetypes know." if sil['valid_roc_auc_archetype_only'].max() - sil['valid_roc_auc_archetype_only'].min() < 0.01
             else "so the choice of k matters for how much the archetypes know."))
print(f"\nUsing {ARCH_K} archetypes")
km = KMeans(n_clusters=ARCH_K, n_init=10, random_state=SEED).fit(Ztr)

# ---- assign EVERY filing (all splits, and the rows S2 will use) to the nearest TRAIN centre
D["f_archetype"] = km.predict(arch_prep.transform(arch_matrix(D))).astype(float)
S["f_archetype"] = D.loc[S.index, "f_archetype"]
tr, va, te = (S[S["split"] == s] for s in ("train", "valid", "test"))      # same rows and order as Cell 6, now with the archetype column

# ---- describe each archetype: its three most distinctive traits, measured in standard deviations from the average training row
cent = pd.DataFrame(km.cluster_centers_, columns=ARCH_FEATURES)
def traits(row, n=3):
    top = row.abs().sort_values(ascending=False).head(n).index
    return ", ".join(f"{'high' if row[c] > 0 else 'low'} {c} ({row[c]:+.1f} sd)" for c in top)
P = tr.assign(archetype=tr["f_archetype"].astype(int))
arch_profile = P.groupby("archetype").agg(
    rows=("adsh", "size"), companies=("cik", "nunique"),
    loss_now=("loss_now", "mean"), loss_next=("loss_next", "mean"),
    median_assets_musd=("Assets", lambda s: s.median() / 1e6), share_spac=("f_is_spac", "mean"),
    top_sector=("sic_division", lambda s: s.mode().iloc[0] if s.notna().any() else "n/a"))
arch_profile.insert(1, "share_of_rows", arch_profile["rows"] / len(P))
arch_profile["traits"] = [traits(cent.loc[i]) for i in arch_profile.index]
print("\nArchetype profiles on the TRAIN rows:")
print(arch_profile.round(3).to_string())

# ---- do the archetypes keep their meaning over time?
SA = S.assign(archetype=S["f_archetype"].astype(int))
print("\nNext-quarter loss rate by archetype and split (similar columns = the archetype means the same thing in the future):")
print(SA.pivot_table(index="archetype", columns="split", values="loss_next", aggfunc="mean")[["train", "valid", "test"]].round(3).to_string())
print("\nShare of rows in each archetype by split (a big shift = the mix of companies is changing):")
print((SA.groupby("split")["archetype"].value_counts(normalize=True).unstack(0)[["train", "valid", "test"]] * 100).round(1).to_string())
prev_arch = D.groupby("cik")["f_archetype"].shift(1).where(prev_ok)
print(f"\nStickiness: {(D['f_archetype'] == prev_arch)[prev_arch.notna()].mean():.1%} of filings keep the archetype of the company's previous true quarter")

# ---- the ablation: does the archetype add anything to the forecast?
CAT_U1 = CAT_COLS + ["archetype"]
H, PS = GROUPS["history"], GROUPS["profit_scale"]
U1_SETS = [("ALL features (the Part 1 model)", FEATURES), ("ALL + archetype", FEATURES + ["archetype"]),
           ("history only", H), ("history + archetype", H + ["archetype"]),
           ("history + profit_scale", H + PS), ("history + profit_scale + archetype", H + PS + ["archetype"]),
           ("archetype only", ["archetype"]), ("loss_now + archetype", ["loss_now", "archetype"])]
u1_rows = []
for label, feats in U1_SETS:
    pv, pt = fit_predict(tr, [va, te], features=feats, cat_cols=CAT_U1)
    u1_rows.append(dict(feature_set=label, n_features=len(feats),
                        valid_roc_auc=roc_auc_score(yva, pv), valid_pr_auc=average_precision_score(yva, pv),
                        test_roc_auc=roc_auc_score(yte, pt), test_pr_auc=average_precision_score(yte, pt)))
u1 = pd.DataFrame(u1_rows)
print("\nU1 ablation (same model and rows; only the feature list changes):")
print(u1.round(4).to_string(index=False))

u1i = u1.set_index("feature_set")["test_roc_auc"]
u1_gain_full = u1i["ALL + archetype"] - u1i["ALL features (the Part 1 model)"]
u1_gain_hist = u1i["history + archetype"] - u1i["history only"]
print(f"\nTest ROC-AUC gained by adding the archetype: to the full model {u1_gain_full:+.4f} | to the history-only model {u1_gain_hist:+.4f}")
u1v = u1.set_index("feature_set")["valid_roc_auc"]
u1_gain_full_valid = u1v["ALL + archetype"] - u1v["ALL features (the Part 1 model)"]
print(f"Validation ROC-AUC gained by adding the archetype to the full model: {u1_gain_full_valid:+.4f} (the decision is made on this number; the test gains above only confirm it)")
print("Verdict:", "the archetype is worth adding as a feature." if u1_gain_full_valid >= 0.002 else
      "the full model already captures what the archetype knows; keep the archetypes as a DESCRIPTION of company types, not as a feature.")

fig, ax = plt.subplots(figsize=(7, 0.45 * ARCH_K + 1.2))
ap = arch_profile.sort_values("loss_next")
ax.barh([f"{i}: {t.split(' (')[0]}" for i, t in zip(ap.index, ap["traits"])], ap["loss_next"], color="#0F766E")
ax.set_xlabel("next-quarter loss rate (train)"); ax.set_title("U1 archetypes: how often each type reports a loss next quarter")
plt.tight_layout(); plt.show()

### Cell 15: a cheaper model? How much of the lift does a small, transparent model keep?
The plan was to look for a cheaper model **if the lift over persistence turned out small**. The cell first checks that condition, then answers the practical question either way: with only the `TOP_K` most important features (Cell 11's ranking on the validation period), how much of the full model's lift over persistence is kept, on the test period and in the transition view?

`share_of_full_lift_kept` = (model's test ROC-AUC − persistence) / (full model − persistence). A small model that keeps most of the lift is easier to explain, cheaper to run and harder to overfit. The coefficients of the small logistic model are printed at the end, so that whole model fits on one screen.

The features are chosen on the validation period, and so is the cheaper model: `valid_share_of_lift_kept` decides, and the test column only confirms. The validation scores of the top-K models are slightly flattering, because the features were ranked on the same period.

**Result of this run:** the full model's lift is not small, so it earns its place. **Gradient boosting on only the top 8 features keeps about 96% of the lift on validation** (ROC-AUC 0.930 against 0.934), and the test period confirms it (95%; 0.919 against 0.924), so it is the cheaper model to use. A logistic regression on the same 8 features keeps only about 46% on validation and 36% on test: the 8 features were ranked by the boosting model, and that ranking leaves out the loss flags a linear model needs.

In [ ]:
SMALL_LIFT = 0.02     # a VALIDATION ROC-AUC lift over persistence below this would make a cheaper model the honest choice
TOP_K = 8             # how many of Cell 11's top features the small models get

pers_auc, pers_pr = get("test", "persistence (next = now)", "roc_auc"), get("test", "persistence (next = now)", "pr_auc")
full_auc = get("test", "gradient boosting", "roc_auc")
s1_lift = full_auc - pers_auc
pers_auc_v, full_auc_v = get("valid", "persistence (next = now)", "roc_auc"), get("valid", "gradient boosting", "roc_auc")
s1_lift_v = full_auc_v - pers_auc_v
print(f"Full model's VALIDATION ROC-AUC lift over persistence: {s1_lift_v:+.4f} (test, for confirmation: {s1_lift:+.4f}) -> " +
      ("SMALL: prefer the cheapest model below that keeps most of it." if s1_lift_v < SMALL_LIFT else
       "NOT small: the full model earns its place. The table still shows how much a small model keeps."))

top_feats = imp_t["feature"].head(TOP_K).tolist()
print(f"\nTop {TOP_K} features (Cell 11, validation period):", ", ".join(top_feats))

def make_logit(features, C=0.5):
    """The Cell 8 logistic pipeline for any feature list (median + missing flag + scaling; one-hot for categoricals)."""
    num = [c for c in features if c not in CAT_COLS]; cat = [c for c in features if c in CAT_COLS]
    parts = []
    if num: parts.append(("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)), ("sc", StandardScaler())]), num))
    if cat: parts.append(("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value=-1)), ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat))
    return Pipeline([("prep", ColumnTransformer(parts)), ("clf", LogisticRegression(C=C, max_iter=2000))])

sel_t = (te["f_loss_now"] == 0).values      # the transition rows of the test period (profitable now)
def cheap_row(label, n_features, pv, pt):
    return dict(model=label, n_features=n_features,
                valid_roc_auc=roc_auc_score(yva, pv), test_roc_auc=roc_auc_score(yte, pt), test_pr_auc=average_precision_score(yte, pt),
                test_transition_roc_auc=roc_auc_score(yte[sel_t], np.asarray(pt)[sel_t]) if len(np.unique(yte[sel_t])) == 2 else np.nan)

cheap = [cheap_row("persistence (next = now)", 1, va["f_loss_now"].values, te["f_loss_now"].values),
         cheap_row("history table (last 2 quarters)", 2, history_prob(va), history_prob(te))]
lean = make_logit(top_feats).fit(Xtr[top_feats], ytr)
cheap.append(cheap_row(f"logistic, top {TOP_K}", TOP_K, lean.predict_proba(Xva[top_feats])[:, 1], lean.predict_proba(Xte[top_feats])[:, 1]))
pv, pt = fit_predict(tr, [va, te], features=top_feats)
cheap.append(cheap_row(f"gradient boosting, top {TOP_K}", TOP_K, pv, pt))
cheap.append(cheap_row("logistic, all (Cell 8)", len(FEATURES), pred[("logistic", "valid")], pred[("logistic", "test")]))
cheap.append(cheap_row("gradient boosting, all (Cell 8)", len(FEATURES), pred[("gradient boosting", "valid")], pred[("gradient boosting", "test")]))
cheap = pd.DataFrame(cheap)
cheap["valid_share_of_lift_kept"] = (cheap["valid_roc_auc"] - pers_auc_v) / (full_auc_v - pers_auc_v)
cheap["share_of_full_lift_kept"] = (cheap["test_roc_auc"] - pers_auc) / (full_auc - pers_auc)
print("\nHow much model is needed? (choose on valid_share_of_lift_kept; share_of_full_lift_kept is the same measure on the test period)")
print(cheap.round(4).to_string(index=False))

kept = cheap.set_index("model")[["valid_share_of_lift_kept", "share_of_full_lift_kept"]]
lg_v, lean_kept = kept.loc[f"logistic, top {TOP_K}"]; gb_v, gb_kept = kept.loc[f"gradient boosting, top {TOP_K}"]
print(f"\nThe {TOP_K}-feature logistic regression keeps {lg_v:.0%} of the full model's lift over persistence on validation ({lean_kept:.0%} on test).")
print(f"Gradient boosting on the same {TOP_K} features keeps {gb_v:.0%} on validation ({gb_kept:.0%} on test): "
      + ("that is the cheaper model to use." if gb_v >= lg_v else "the logistic model is the cheaper model to use."))

coef = pd.Series(lean.named_steps["clf"].coef_[0], index=lean.named_steps["prep"].get_feature_names_out())
coef.index = [c.replace("num__", "").replace("cat__", "") for c in coef.index]
print(f"\nThe whole {TOP_K}-feature logistic model (standardised coefficients; positive = more likely to report a loss next quarter;"
      " 'missingindicator_x' = the value was blank):")
print(coef.reindex(coef.abs().sort_values(ascending=False).index).round(3).to_string())
print(f"intercept: {lean.named_steps['clf'].intercept_[0]:.3f}")

### Cell 16a: S2, equity turns negative. The label, the rows, the split and the baselines
**Question.** Among companies with **positive total equity** in their latest filing, which ones will report **negative equity** (liabilities larger than assets) at their next true quarter? It is a balance-sheet distress signal, much rarer than a quarterly loss. With so few events (about 140 in the test period), S2 is an **exploratory screening model**: its scores carry wide uncertainty, which Cell 19 measures.

**Same machinery as S1:** the same "true next quarter" rule (75 to 110 days), the same row filters, the same time split on the label quarter, the same 30 features and the same `fit_predict`. There is one extra feature, built only from the current filing:
- `eq_cushion_q`: how many quarters of the *current* loss the equity could absorb (equity ÷ quarterly loss, capped at 40; profitable now = 40).

**Why only positive-equity rows:** the sign of equity is very sticky (the persistence table below shows it). Predicting that a company with negative equity stays negative is trivial. The useful question is the *transition*.

**Baselines (ranking rules, no model):** *thin equity* ranks by the lowest equity ÷ assets; *burn rate* ranks by the fewest quarters of cushion; *loss now* flags every company losing money. Rare events are judged by **PR-AUC** and by **top-10% capture** (the share of all events found in the 10% of filings the model ranks most at risk), not by accuracy.

In [ ]:
# ---------------- the S2 label: total equity at the NEXT true quarter (same rule as the S1 label in Cell 5) ----------------
D["eq_next"] = g["EquityTotal"].shift(-1).where(next_ok)

# ---------------- one extra feature, from the CURRENT filing only: quarters of the current loss the equity can absorb ----------------
EQ_CUSHION_CAP = 40
E = D["EquityTotal"]
cushion = np.where(D["ni_q"] < 0, E / (-D["ni_q"]), np.where(D["ni_q"] >= 0, EQ_CUSHION_CAP, np.nan))
D["f_eq_cushion_q"] = pd.Series(cushion, index=D.index).clip(0, EQ_CUSHION_CAP).where(E.notna())

# ---------------- rows: the same filters as Cell 6, with the S2 label instead of the S1 label ----------------
funnel2 = []
m2 = pd.Series(True, index=D.index);                  funnel2.append(("all filings", int(m2.sum())))
m2 &= D["EquityTotal"].notna();                       funnel2.append(("+ total equity reported this quarter", int(m2.sum())))
m2 &= D["eq_next"].notna();                           funnel2.append(("+ a true next quarter with total equity (the label)", int(m2.sum())))
m2 &= D["period"] >= pd.Timestamp(WINDOW_START);      funnel2.append((f"+ period on or after {WINDOW_START}", int(m2.sum())))
m2 &= D["days_to_file"].between(0, MAX_DAYS_TO_FILE); funnel2.append((f"+ days_to_file between 0 and {MAX_DAYS_TO_FILE}", int(m2.sum())))
m2 &= D["has_negative_nonneg_tag"].fillna(0) == 0;    funnel2.append(("+ no negative value in a never-negative tag", int(m2.sum())))
S2all = D[m2].copy()
S2all["neg_eq_now"] = (S2all["EquityTotal"] < 0).astype(float)
S2all["neg_eq_next"] = (S2all["eq_next"] < 0).astype(float)
S2all["split"] = np.where(S2all["next_period"] <= pd.Timestamp(TRAIN_END), "train",
                          np.where(S2all["next_period"] <= pd.Timestamp(VALID_END), "valid", "test"))
S2all["segment"] = np.where(S2all["f_is_spac"] == 1, "SPAC", np.where(S2all["f_is_shell"] == 1, "shell", "operating company"))
print(pd.DataFrame(funnel2, columns=["step", "rows left"]).to_string(index=False))

print("\nHow sticky is the sign of equity? P(negative equity next quarter), by this quarter's sign and split:")
print(S2all.assign(now=np.where(S2all["neg_eq_now"] == 1, "negative now", "positive or zero now"))
      .pivot_table(index="now", columns="split", values="neg_eq_next", aggfunc="mean")[["train", "valid", "test"]].round(4))

# ---------------- the S2 population: positive equity NOW ----------------
S2 = S2all[S2all["EquityTotal"] > 0].copy()
summary2 = S2.groupby("split").agg(rows=("adsh", "size"), companies=("cik", "nunique"), events=("neg_eq_next", "sum"),
                                   event_rate=("neg_eq_next", "mean"), first_label_quarter=("next_period", "min"),
                                   last_label_quarter=("next_period", "max")).loc[["train", "valid", "test"]]
print("\nS2 population (positive equity now). An event = negative equity at the next true quarter:\n", summary2)
print("\nEvent rate by segment and split (SPACs often show negative equity because redeemable shares sit outside equity):")
print(S2.pivot_table(index="segment", columns="split", values="neg_eq_next", aggfunc=["mean", "sum"]).round(4).to_string())

tr2, va2, te2 = (S2[S2["split"] == s] for s in ("train", "valid", "test"))
ytr2, yva2, yte2 = (df["neg_eq_next"].values for df in (tr2, va2, te2))
assert ytr2.sum() >= 50 and len(np.unique(yva2)) == 2 and len(np.unique(yte2)) == 2, "too few S2 events in a split: move TRAIN_END / VALID_END"
if min(yva2.sum(), yte2.sum()) < 30:
    print("\nCAUTION: fewer than 30 events in the validation or test period; read the S2 scores as rough.")

# ---------------- ranking metrics for a rare event ----------------
def rank_metrics(y, s, top=0.10):
    y = np.asarray(y, dtype=float); s = np.asarray(s, dtype=float)
    n_top = max(1, int(round(top * len(y))))
    hit = y[np.argsort(-s, kind="mergesort")[:n_top]]
    return dict(n=len(y), events=int(y.sum()), event_rate=y.mean(), roc_auc=roc_auc_score(y, s), pr_auc=average_precision_score(y, s),
                top10_precision=hit.mean(), top10_capture=hit.sum() / max(y.sum(), 1))

# ---------------- baselines: three ranking rules learned from nothing (blanks filled with the TRAIN median) ----------------
eqr_med, cush_med = tr2["f_equity_ratio"].median(), tr2["f_eq_cushion_q"].median()
S2_BASELINES = {
    "thin equity (lowest equity / assets)": lambda df: -df["f_equity_ratio"].fillna(eqr_med).values,
    "burn rate (fewest quarters of cushion)": lambda df: -df["f_eq_cushion_q"].fillna(cush_med).values,
    "loss now": lambda df: df["f_loss_now"].fillna(0).values,
}
s2_rows = []
for split, df, y in (("valid", va2, yva2), ("test", te2, yte2)):
    s2_rows.append(dict(split=split, model="majority (train rate)", **rank_metrics(y, np.full(len(y), ytr2.mean()))))
    for name, rule in S2_BASELINES.items():
        s2_rows.append(dict(split=split, model=name, **rank_metrics(y, rule(df))))
s2_baselines = pd.DataFrame(s2_rows)
print("\nS2 baselines:")
print(s2_baselines.round(4).to_string(index=False))

### Cell 16b: S2 models, lift over the best rule, slices and rolling origin
Gradient boosting through the same `fit_predict` (target switched to `neg_eq_next`) and the same logistic pipeline, with an ablation for the new cushion feature and for the U1 archetype. **Lift** is measured against the *best* of the three rules on each metric, so the model has to beat the strongest simple competitor. The rolling-origin check repeats the train-then-test step quarter by quarter, as Cell 12b does for S1.

In [ ]:
S2_FEATURES = FEATURES + ["eq_cushion_q"]

def prep_rows(df, features, train_df):
    """Cell 8's preparation for any frame: clip limits from the TRAINING rows only."""
    cc = ["f_" + c for c in CLIP_COLS if c in features]
    lo_, hi_ = train_df[cc].quantile(0.01), train_df[cc].quantile(0.99)
    X = df[["f_" + c for c in features]].copy()
    X[cc] = X[cc].clip(lower=lo_, upper=hi_, axis=1)
    X.columns = features
    return X

# ---- the models (and an ablation for the two additions)
S2_SETS = [("gradient boosting, S1 features + cushion", S2_FEATURES, CAT_COLS),
           ("gradient boosting, S1 features only", FEATURES, CAT_COLS),
           ("gradient boosting, + cushion + archetype", S2_FEATURES + ["archetype"], CAT_U1)]
pred2 = {}
for label, feats, cats in S2_SETS:
    pred2[(label, "valid")], pred2[(label, "test")] = fit_predict(tr2, [va2, te2], features=feats, target="neg_eq_next", cat_cols=cats)
logit2 = make_logit(S2_FEATURES).fit(prep_rows(tr2, S2_FEATURES, tr2), ytr2)
pred2[("logistic, S1 features + cushion", "valid")] = logit2.predict_proba(prep_rows(va2, S2_FEATURES, tr2))[:, 1]
pred2[("logistic, S1 features + cushion", "test")] = logit2.predict_proba(prep_rows(te2, S2_FEATURES, tr2))[:, 1]

model_rows2 = [dict(split=s, model=m, **rank_metrics(y, pred2[(m, s)]))
               for s, y in (("valid", yva2), ("test", yte2)) for m in sorted({k[0] for k in pred2})]
s2_results = pd.concat([s2_baselines, pd.DataFrame(model_rows2)], ignore_index=True).sort_values(["split", "pr_auc"], ascending=[True, False])
print("S2 results (sorted by PR-AUC, the main score for a rare event):")
print(s2_results.round(4).to_string(index=False))

S2_MAIN = "gradient boosting, S1 features + cushion"
rules = list(S2_BASELINES)
lift2 = []
for s in ("valid", "test"):
    r = s2_results[s2_results["split"] == s].set_index("model")
    for metric in ("roc_auc", "pr_auc", "top10_capture"):
        best_rule = r.loc[rules, metric].idxmax()
        lift2.append(dict(split=s, metric=metric, model=r.loc[S2_MAIN, metric], best_rule=r.loc[best_rule, metric],
                          lift=r.loc[S2_MAIN, metric] - r.loc[best_rule, metric], best_rule_name=best_rule))
lift2 = pd.DataFrame(lift2)
print(f"\nLIFT of '{S2_MAIN}' over the BEST rule on each metric (positive = the model adds something):")
print(lift2.round(4).to_string(index=False))

# ---- slices: is the skill only in SPACs (where negative equity is an accounting convention)?
p2t = pd.Series(pred2[(S2_MAIN, "test")], index=te2.index)
seg_rows = []
for seg, g_ in te2.groupby("segment"):
    yy = g_["neg_eq_next"].values
    if len(g_) >= 100 and len(np.unique(yy)) == 2:
        seg_rows.append(dict(segment=seg, rows=len(g_), events=int(yy.sum()), event_rate=yy.mean(),
                             roc_auc=roc_auc_score(yy, p2t.loc[g_.index]), pr_auc=average_precision_score(yy, p2t.loc[g_.index]),
                             burn_rate_rule_pr_auc=average_precision_score(yy, S2_BASELINES["burn rate (fewest quarters of cushion)"](g_))))
print("\nTEST scores of the main S2 model by segment:")
print(pd.DataFrame(seg_rows).round(4).to_string(index=False))

# ---- rolling origin, as Cell 12b
S2["label_q"] = S2["next_period"].dt.to_period("Q")
ROLLING_MIN_EVENTS2 = 15
roll2 = []
for q in sorted(S2["label_q"].unique()):
    if q < pd.Period(ROLLING_FIRST_TEST):
        continue
    trn, tst = S2[S2["label_q"] < q], S2[S2["label_q"] == q]
    y = tst["neg_eq_next"].values
    if len(trn) < ROLLING_MIN_TRAIN or y.sum() < ROLLING_MIN_EVENTS2 or len(np.unique(y)) < 2:
        continue
    p, = fit_predict(trn, [tst], features=S2_FEATURES, target="neg_eq_next")
    med = trn["f_eq_cushion_q"].median()
    rule = -tst["f_eq_cushion_q"].fillna(med).values
    roll2.append(dict(label_quarter=str(q), train_rows=len(trn), test_rows=len(tst), events=int(y.sum()),
                      gbm_pr_auc=average_precision_score(y, p), burn_rule_pr_auc=average_precision_score(y, rule),
                      gbm_roc_auc=roc_auc_score(y, p), burn_rule_roc_auc=roc_auc_score(y, rule)))
roll2 = pd.DataFrame(roll2)
if len(roll2):
    roll2["pr_auc_lift"] = roll2["gbm_pr_auc"] - roll2["burn_rule_pr_auc"]
    print("\nS2 rolling origin (each quarter tested on a model trained only on earlier quarters):")
    print(roll2.round(3).to_string(index=False))
    print(f"PR-AUC lift over the burn-rate rule: mean {roll2['pr_auc_lift'].mean():+.3f}, quarters where the model did NOT beat it: "
          f"{int((roll2['pr_auc_lift'] <= 0).sum())} of {len(roll2)}")
else:
    print(f"\nS2 rolling origin skipped: no quarter after {ROLLING_FIRST_TEST} has {ROLLING_MIN_EVENTS2}+ events.")

### Cell 16c: S2, what drives it, and the biggest misses by company name
Permutation importance on the validation period (as Cell 11), then two lists from the test period: **missed events** (equity turned negative, the model gave a low risk) and **false alarms** (high risk, equity stayed positive). As with S1, open a few filings. A large one-off write-down is a real surprise. A tiny equity figure that flips on a reclassification is closer to a data quirk.

**What the largest misses were (checked against public filings and company releases).** They are real corporate actions, not data errors:
- **Coca-Cola Consolidated** bought back all the shares held by The Coca-Cola Company on 7 Nov 2025: 18.8 million shares at USD 127, about USD 2.4 billion, partly funded with a term loan. The company was profitable with a full equity cushion, so a low risk score was the right call on the information in its filings.
- **Vivid Seats** booked a USD 402.6 million goodwill and trademark impairment in Q4 2025, more than its equity at the start of the quarter.
- **Sensei Biotherapeutics** acquired Faeth Therapeutics in February 2026 together with a USD 200 million private placement. The event is confirmed; the reported equity of about −USD 137 million still needs checking against the Q1 2026 10-Q.

A filings-only model cannot see such actions before they are announced. That is a limit of the inputs, not a bug in the pipeline.

In [ ]:
(_, ), s2_model, s2_prep = fit_predict(tr2, [va2], features=S2_FEATURES, target="neg_eq_next", return_model=True)
imp2 = permutation_importance(s2_model, s2_prep(va2), yva2, scoring="average_precision", n_repeats=3, random_state=SEED, n_jobs=1)
imp2_t = pd.DataFrame({"feature": S2_FEATURES, "pr_auc_drop": imp2.importances_mean, "std": imp2.importances_std}).sort_values("pr_auc_drop", ascending=False)
print("S2 permutation importance (drop in validation PR-AUC when one feature is shuffled):")
print(imp2_t.head(12).round(4).to_string(index=False))

cols2 = ["name", "form", "segment", "period", "next_period", "EquityTotal", "eq_next", "ni_q", "f_eq_cushion_q"]
err2 = te2[cols2].assign(p_neg_eq=p2t.values, neg_eq_next=yte2)
fmt = lambda df: df.rename(columns={"f_eq_cushion_q": "cushion_q"}).round(3).to_string(index=False)
print("\nMISSED events on the test period (equity turned negative, lowest predicted risk first):")
print(fmt(err2[err2["neg_eq_next"] == 1].sort_values("p_neg_eq").head(10)))
print("\nFALSE ALARMS on the test period (equity stayed positive, highest predicted risk first):")
print(fmt(err2[err2["neg_eq_next"] == 0].sort_values("p_neg_eq", ascending=False).head(10)))

### Cell 17: save Part 2 and print every headline number in one table
Files land in `/kaggle/working`, so they appear in the notebook's **Output** tab after *Save Version*. The headline table is built from the variables above, so it is never out of date.

In [ ]:
OUT_DIR = "/kaggle/working"
os.makedirs(OUT_DIR, exist_ok=True)

# ---- files
S.assign(archetype=S["f_archetype"].astype(int))[["adsh", "cik", "name", "period", "split", "archetype"]].to_parquet(f"{OUT_DIR}/u1_archetypes.parquet", index=False)
arch_profile.to_csv(f"{OUT_DIR}/u1_archetype_profiles.csv")
u1.round(5).to_csv(f"{OUT_DIR}/u1_ablation.csv", index=False)
cheap.round(5).to_csv(f"{OUT_DIR}/cheaper_model.csv", index=False)
s2_results.round(5).to_csv(f"{OUT_DIR}/s2_metrics.csv", index=False)
te2[["adsh", "cik", "name", "segment", "period", "next_period", "neg_eq_next"]].assign(
    p_gbm=pred2[(S2_MAIN, "test")], p_logistic=pred2[("logistic, S1 features + cushion", "test")]).to_parquet(f"{OUT_DIR}/s2_test_predictions.parquet", index=False)

# ---- one table with every headline number
trans = pd.DataFrame(trans_rows).set_index("model")
r2 = s2_results[s2_results["split"] == "test"].set_index("model")
best_rule_pr = lift2[(lift2["split"] == "test") & (lift2["metric"] == "pr_auc")].iloc[0]
# the cheaper-model rows: gradient boosting on the top features (the model to use), with the logistic model on the same features beside it
gb8 = cheap.loc[cheap["model"] == f"gradient boosting, top {TOP_K}"].iloc[0]
lg8 = cheap.loc[cheap["model"] == f"logistic, top {TOP_K}"].iloc[0]
headline = pd.DataFrame([
    ("S1 next-quarter loss", "test ROC-AUC: gradient boosting | persistence", f"{full_auc:.3f} | {pers_auc:.3f}"),
    ("S1 next-quarter loss", "test PR-AUC: gradient boosting | persistence", f"{get('test', 'gradient boosting', 'pr_auc'):.3f} | {pers_pr:.3f}"),
    ("S1 next-quarter loss", "rolling origin: mean ROC-AUC lift, quarters won", f"{roll['roc_auc_lift'].mean():+.3f}, {int((roll['roc_auc_lift'] > 0).sum())} of {len(roll)}"),
    ("S1 transition (profitable now)", "test ROC-AUC | PR-AUC, gradient boosting",
     f"{trans.loc['gradient boosting', 'roc_auc']:.3f} | {trans.loc['gradient boosting', 'pr_auc']:.3f} (turn-to-loss rate {trans.loc['gradient boosting', 'turn_to_loss_rate']:.3f})"),
    ("U1 archetypes", "number of archetypes | stickiness", f"{ARCH_K} | {(D['f_archetype'] == prev_arch)[prev_arch.notna()].mean():.1%}"),
    ("U1 archetypes", "test ROC-AUC gain: added to the full model | to history only", f"{u1_gain_full:+.4f} | {u1_gain_hist:+.4f}"),
    ("Cheaper model", f"gradient boosting on {TOP_K} features: test ROC-AUC, share of lift kept",
     f"{gb8['test_roc_auc']:.3f}, {gb8['share_of_full_lift_kept']:.0%}"),
    ("Cheaper model", f"logistic on the same {TOP_K} features (for comparison)",
     f"{lg8['test_roc_auc']:.3f}, {lg8['share_of_full_lift_kept']:.0%}"),
    ("S2 equity turns negative", "test rows, events (rate)", f"{len(te2):,}, {int(yte2.sum())} ({yte2.mean():.2%})"),
    ("S2 equity turns negative", "test PR-AUC: model | best rule", f"{r2.loc[S2_MAIN, 'pr_auc']:.3f} | {best_rule_pr['best_rule']:.3f} ({best_rule_pr['best_rule_name']})"),
    ("S2 equity turns negative", "test ROC-AUC | top-10% capture, model", f"{r2.loc[S2_MAIN, 'roc_auc']:.3f} | {r2.loc[S2_MAIN, 'top10_capture']:.1%}"),
], columns=["part", "measure", "value"])
headline.to_csv(f"{OUT_DIR}/headline_numbers.csv", index=False)
print(headline.to_string(index=False))
print("\nsaved to", OUT_DIR, ":", ", ".join(sorted(f for f in os.listdir(OUT_DIR) if f.startswith(("u1_", "s2_", "cheaper", "headline")))))

---
# Part 3: how far to trust the results, and how to use them

Parts 1 and 2 built and tested the models. Part 3 asks the questions a reviewer asks before anyone relies on them:

| Cell | Step | Question |
|---|---|---|
| 18 | **Filing-date validation** | Do the results hold when the model may only learn from answers that were already public on the day of each forecast? |
| 19 | **Confidence intervals** | How much could the headline scores move with a different set of companies? |
| 20 | **Decision thresholds** | Which probability cut-off turns a score into a yes/no flag, chosen on validation data only? |
| 21 | **Later-period and unseen-company tests** | Does the model keep working further away from its training data, and on companies it has never seen? |
| 22 | **Model card** | What is the model for, what is it not for, and what are its risks? |

### Cell 18: filing-date validation (a strictly prospective test)
**Why.** Cell 6 splits by the *label quarter*. That keeps every training answer earlier than every test answer, but it does not check **when the answers became public**. A training row's answer is public only once the company files its *next* report, which can be weeks or months after that quarter ends. Near the split dates, a model trained on "label quarters up to December 2024" uses 10-Ks filed in February and March 2025, after some validation forecasts were already due.

**What.** Each row gets two dates:
- `filed_date` = period end + `days_to_file`: when the features became public;
- `label_known_date` = the next quarter's period end + that filing's `days_to_file`: when the answer became public.

At a forecast date *T* the model may only train on rows whose answer was public **before T**, and it is scored on the filings made **on or after T**. The forecast dates are the same calendar dates as Cell 6 (1 January 2025 for validation, 1 October 2025 for test). Both protocols use the same `fit_predict`, logistic and history-table recipes, so the only difference is which rows train and which rows are scored.

**How to read it.** First, how many training answers under the label-quarter split were not yet public at the forecast dates. Then the two protocols side by side, a walk-forward version (retrain at the start of every filing quarter on what was public then, which is how the model would actually be run), and the same check for S2. In the most recent filing quarters only rows whose next report has already arrived can be scored, so those quarters lean towards prompt filers.

In [ ]:
# ---- Part 3 settings: the forecast dates are the SAME calendar dates as Cell 6, applied to filing dates instead of label quarters
FD_VALID_START = pd.Timestamp(TRAIN_END) + pd.Timedelta(days=1)     # validation forecasts start here
FD_TEST_START = pd.Timestamp(VALID_END) + pd.Timedelta(days=1)       # test forecasts start here

# ---- two dates per row: when the FEATURES became public, and when the ANSWER (the label) became public
D["filed_date"] = D["period"] + pd.to_timedelta(D["days_to_file"], unit="D")
D["label_known_date"] = D["next_period"] + pd.to_timedelta(g["days_to_file"].shift(-1).where(next_ok), unit="D")
for frame in (S, S2):
    frame["filed_date"] = D.loc[frame.index, "filed_date"]
    frame["label_known_date"] = D.loc[frame.index, "label_known_date"]
tr, va, te = (S[S["split"] == s] for s in ("train", "valid", "test"))          # same rows and order, now with the two dates
tr2, va2, te2 = (S2[S2["split"] == s] for s in ("train", "valid", "test"))
print(f"S1 rows: filed {S['filed_date'].min().date()} to {S['filed_date'].max().date()}; answers public up to {S['label_known_date'].max().date()}")
print(f"S1 rows whose answer date is unknown (the next filing has no delay figure): {int(S['label_known_date'].isna().sum()):,}; they are never used for training here")

# ---- 1. under the label-quarter split, how many training answers were NOT yet public when the forecasts were due?
print("\nThe label-quarter split (Cell 6) seen on the calendar:")
for name, df, when in (("TRAIN", tr, FD_VALID_START), ("TRAIN", tr, va["filed_date"].min()), ("TRAIN", tr, FD_TEST_START)):
    late = df["label_known_date"] >= when
    print(f"  {name} rows whose answer became public on or after {when.date()}: {int(late.sum()):,} of {len(df):,} ({late.mean():.1%})")
print(f"  (the first VALIDATION filing is dated {va['filed_date'].min().date()}, the first TEST filing {te['filed_date'].min().date()})")

# ---- 2. the same models under the filing-date protocol
def history_rule(train_df, min_size=50):
    """Cell 7's history table, learned from any training frame: P(loss next | loss now, loss last quarter)."""
    a, b = train_df["f_loss_now"], train_df["f_loss_lag1"].fillna(-1)
    h = train_df.groupby([a, b])["loss_next"].agg(["mean", "size"])
    h = h.loc[h["size"] >= min_size, "mean"]
    now = train_df.groupby(a)["loss_next"].mean()
    def prob(df):
        idx = pd.MultiIndex.from_arrays([df["f_loss_now"].values, df["f_loss_lag1"].fillna(-1).values])
        p = h.reindex(idx).values
        return np.where(np.isnan(p), now.reindex(df["f_loss_now"].values).values, p)
    return prob
assert np.allclose(history_rule(tr)(te), history_prob(te)), "history_rule must reproduce Cell 7's history table"

S1_MODELS = ["gradient boosting", "logistic", "history table (last 2 quarters)", "persistence (next = now)"]
def s1_scores(train_df, eval_df):
    """The four S1 scores for eval_df, each learned from train_df only (the fit_predict and logistic recipes of Parts 1 and 2)."""
    p_gbm, = fit_predict(train_df, [eval_df])
    lg = make_logit(FEATURES).fit(prep_rows(train_df, FEATURES, train_df), train_df["loss_next"].values)
    return pd.DataFrame({"gradient boosting": p_gbm,
                         "logistic": lg.predict_proba(prep_rows(eval_df, FEATURES, train_df))[:, 1],
                         "history table (last 2 quarters)": history_rule(train_df)(eval_df),
                         "persistence (next = now)": eval_df["f_loss_now"].values}, index=eval_df.index)

def s1_rows(scores, df, train_df, protocol, split):
    y = df["loss_next"].values
    sc = scores.loc[df.index]
    pers_roc, pers_pr = roc_auc_score(y, sc["persistence (next = now)"]), average_precision_score(y, sc["persistence (next = now)"])
    return [dict(protocol=protocol, split=split, model=m, train_rows=len(train_df), rows=len(df), loss_rate=y.mean(),
                 roc_auc=roc_auc_score(y, sc[m]), pr_auc=average_precision_score(y, sc[m]),
                 roc_auc_lift=roc_auc_score(y, sc[m]) - pers_roc, pr_auc_lift=average_precision_score(y, sc[m]) - pers_pr) for m in S1_MODELS]

fd_train_v = S[S["label_known_date"] < FD_VALID_START]                                   # answers public before the validation date
fd_valid = S[(S["filed_date"] >= FD_VALID_START) & (S["filed_date"] < FD_TEST_START)]   # filed in the validation window
fd_train_t = S[S["label_known_date"] < FD_TEST_START]                                    # answers public before the test date
fd_test = S[S["filed_date"] >= FD_TEST_START]                                            # filed on or after the test date
fd_later = S[S["filed_date"] >= FD_VALID_START]                                          # everything after the first date (Cell 21)

lq_scores = s1_scores(tr, pd.concat([va, te]))           # label-quarter protocol, same recipe
fd_frozen = s1_scores(fd_train_v, fd_later)              # filing-date protocol, model fixed at the validation date
fd_retrained = s1_scores(fd_train_t, fd_test)            # filing-date protocol, model retrained at the test date
fd_compare = pd.DataFrame(
    s1_rows(lq_scores, va, tr, "label quarter (Cell 6)", "valid") +
    s1_rows(lq_scores, te, tr, "label quarter (Cell 6)", "test") +
    s1_rows(fd_frozen, fd_valid, fd_train_v, "filing date", "valid") +
    s1_rows(fd_frozen, fd_test, fd_train_v, "filing date", "test, model fixed at the validation date") +
    s1_rows(fd_retrained, fd_test, fd_train_t, "filing date", "test, retrained at the test date"))
print("\nThe same recipes under both protocols (lift = minus persistence on the same rows):")
print(fd_compare.round(4).to_string(index=False))

gb = fd_compare[fd_compare["model"] == "gradient boosting"].set_index(["protocol", "split"])
lq_t, fd_t = gb.loc[("label quarter (Cell 6)", "test")], gb.loc[("filing date", "test, retrained at the test date")]
lq_v, fd_v = gb.loc[("label quarter (Cell 6)", "valid")], gb.loc[("filing date", "valid")]
print(f"\nGradient boosting, validation: label quarter {lq_v['roc_auc']:.4f} | filing date {fd_v['roc_auc']:.4f} (difference {fd_v['roc_auc'] - lq_v['roc_auc']:+.4f})")
print(f"Gradient boosting, test:       label quarter {lq_t['roc_auc']:.4f} | filing date {fd_t['roc_auc']:.4f} (difference {fd_t['roc_auc'] - lq_t['roc_auc']:+.4f})")
print(f"Lift over persistence under the filing-date protocol: validation {fd_v['roc_auc_lift']:+.4f}, test {fd_t['roc_auc_lift']:+.4f}")
fd_ok = (fd_v["roc_auc_lift"] > 0) and (fd_t["roc_auc_lift"] > 0) and abs(fd_t["roc_auc"] - lq_t["roc_auc"]) < 0.01
print("Verdict:", "the results hold under the strictly prospective filing-date protocol." if fd_ok else
      "the filing-date protocol changes the picture: read the table above before relying on Part 1's numbers.")

# ---- 3. walk-forward by FILING quarter: retrain at the start of each quarter on the answers public by then
S["filed_q"] = S["filed_date"].dt.to_period("Q")
roll_fd = []
for q in sorted(S["filed_q"].dropna().unique()):
    if q < pd.Period(ROLLING_FIRST_TEST):
        continue
    trn, tst = S[S["label_known_date"] < q.start_time], S[S["filed_q"] == q]
    y = tst["loss_next"].values
    if len(trn) < ROLLING_MIN_TRAIN or len(tst) < ROLLING_MIN_TEST or len(np.unique(y)) < 2:
        continue
    p, = fit_predict(trn, [tst])
    pers = tst["f_loss_now"].values
    roll_fd.append(dict(filing_quarter=str(q), train_rows=len(trn), test_rows=len(tst), loss_rate=y.mean(),
                        gbm_roc_auc=roc_auc_score(y, p), persistence_roc_auc=roc_auc_score(y, pers),
                        gbm_pr_auc=average_precision_score(y, p), persistence_pr_auc=average_precision_score(y, pers)))
roll_fd = pd.DataFrame(roll_fd)
roll_fd["roc_auc_lift"] = roll_fd["gbm_roc_auc"] - roll_fd["persistence_roc_auc"]
roll_fd["pr_auc_lift"] = roll_fd["gbm_pr_auc"] - roll_fd["persistence_pr_auc"]
print("\nWalk-forward by filing quarter (each quarter scored by a model trained only on answers public at its start):")
print(roll_fd.round(3).to_string(index=False))
print(f"ROC-AUC lift over persistence: mean {roll_fd['roc_auc_lift'].mean():+.3f} (min {roll_fd['roc_auc_lift'].min():+.3f}); "
      f"quarters won: {int((roll_fd['roc_auc_lift'] > 0).sum())} of {len(roll_fd)}")

# ---- 4. the same check for S2 (positive equity now -> negative equity next quarter)
fd2_train, fd2_test = S2[S2["label_known_date"] < FD_TEST_START], S2[S2["filed_date"] >= FD_TEST_START]
p2_fd, = fit_predict(fd2_train, [fd2_test], features=S2_FEATURES, target="neg_eq_next")
burn_fd = -fd2_test["f_eq_cushion_q"].fillna(fd2_train["f_eq_cushion_q"].median()).values
r2t = s2_results[s2_results["split"] == "test"].set_index("model")
burn_name = "burn rate (fewest quarters of cushion)"
fd2_compare = pd.DataFrame([
    dict(protocol="label quarter (Cell 16b)", model=S2_MAIN, **r2t.loc[S2_MAIN, ["n", "events", "roc_auc", "pr_auc", "top10_capture"]].to_dict()),
    dict(protocol="label quarter (Cell 16b)", model=burn_name, **r2t.loc[burn_name, ["n", "events", "roc_auc", "pr_auc", "top10_capture"]].to_dict()),
    dict(protocol="filing date", model=S2_MAIN, **{k: v for k, v in rank_metrics(fd2_test["neg_eq_next"].values, p2_fd).items()
                                                   if k in ("n", "events", "roc_auc", "pr_auc", "top10_capture")}),
    dict(protocol="filing date", model=burn_name, **{k: v for k, v in rank_metrics(fd2_test["neg_eq_next"].values, burn_fd).items()
                                                     if k in ("n", "events", "roc_auc", "pr_auc", "top10_capture")})])
fd2_compare[["n", "events"]] = fd2_compare[["n", "events"]].astype(int)
print("\nS2 test period under both protocols:")
print(fd2_compare.round(4).to_string(index=False))

### Cell 19: confidence intervals for the headline scores
**Why.** A single test score does not say how much it could move. Filings of the same company are correlated (a loss-making company tends to stay loss-making), so treating 10,000 filings as 10,000 independent draws would make the scores look more precise than they are.

**How.** A **company-clustered bootstrap**: draw companies with replacement (every filing of a drawn company comes along), recompute each score on the same predictions, repeat `N_BOOT` = 500 times, and report the 2.5th and 97.5th percentiles. The models are not refitted, so the interval covers *which companies happen to be in the test period*; the GroupKFold spread in Cell 8 covers training randomness. For the lifts, `share_above_0` is the share of resamples in which the model beat its baseline.

In [ ]:
N_BOOT = 500          # bootstrap resamples (more = smoother interval ends, slower)

def cluster_bootstrap(df, stats, n_boot=N_BOOT, seed=SEED):
    """Resample COMPANIES with replacement (every filing of a drawn company comes along) and recompute each statistic.
    stats maps a name to a function of a row-position array. The models are not refitted: the interval covers which
    companies happen to be in the evaluation period, not training randomness (Cell 8's GroupKFold covers that)."""
    codes, uniq = pd.factorize(df["cik"])
    order = np.argsort(codes, kind="mergesort")
    bounds = np.searchsorted(codes[order], np.arange(len(uniq) + 1))
    groups = [order[bounds[i]:bounds[i + 1]] for i in range(len(uniq))]
    rng = np.random.default_rng(seed)
    point = {k: f(np.arange(len(df))) for k, f in stats.items()}
    draws = {k: [] for k in stats}
    for _ in range(n_boot):
        idx = np.concatenate([groups[i] for i in rng.integers(0, len(uniq), len(uniq))])
        for k, f in stats.items():
            try:
                draws[k].append(f(idx))
            except ValueError:                      # a resample with only one class
                draws[k].append(np.nan)
    return point, {k: np.asarray(v, dtype=float) for k, v in draws.items()}

def top_capture(y, s, top=0.10):
    n_top = max(1, int(round(top * len(y))))
    return y[np.argsort(-s, kind="mergesort")[:n_top]].sum() / max(y.sum(), 1)

ci_parts = []
def add_ci(block, df, stats, lifts=()):
    point, draws = cluster_bootstrap(df, stats)
    for k in stats:
        lo_, hi_ = np.nanpercentile(draws[k], [2.5, 97.5])
        ci_parts.append(dict(block=block, measure=k, estimate=point[k], ci_low=lo_, ci_high=hi_,
                             share_above_0=np.nanmean(draws[k] > 0) if k in lifts else np.nan,
                             rows=len(df), companies=df["cik"].nunique()))

# ---- S1, label-quarter test (the Cell 8 models, the headline numbers)
y1 = yte.astype(float); p1 = pred[("gradient boosting", "test")]; s1p = te["f_loss_now"].values.astype(float); prof = (te["f_loss_now"] == 0).values
add_ci("S1 test (label quarter)", te, {
    "ROC-AUC, gradient boosting": lambda i: roc_auc_score(y1[i], p1[i]),
    "ROC-AUC, persistence": lambda i: roc_auc_score(y1[i], s1p[i]),
    "ROC-AUC lift over persistence": lambda i: roc_auc_score(y1[i], p1[i]) - roc_auc_score(y1[i], s1p[i]),
    "PR-AUC, gradient boosting": lambda i: average_precision_score(y1[i], p1[i]),
    "PR-AUC lift over persistence": lambda i: average_precision_score(y1[i], p1[i]) - average_precision_score(y1[i], s1p[i]),
    "transition ROC-AUC (profitable now)": lambda i: roc_auc_score(y1[i][prof[i]], p1[i][prof[i]]),
}, lifts=("ROC-AUC lift over persistence", "PR-AUC lift over persistence"))

# ---- S1, filing-date test (Cell 18, retrained at the test date)
yf = fd_test["loss_next"].values.astype(float); pf = fd_retrained.loc[fd_test.index, "gradient boosting"].values; sf = fd_test["f_loss_now"].values.astype(float)
add_ci("S1 test (filing date)", fd_test, {
    "ROC-AUC, gradient boosting": lambda i: roc_auc_score(yf[i], pf[i]),
    "ROC-AUC lift over persistence": lambda i: roc_auc_score(yf[i], pf[i]) - roc_auc_score(yf[i], sf[i]),
}, lifts=("ROC-AUC lift over persistence",))

# ---- S2 test (Cell 16b)
y2 = yte2.astype(float); p2 = pred2[(S2_MAIN, "test")]; b2 = S2_BASELINES["burn rate (fewest quarters of cushion)"](te2)
add_ci("S2 test (label quarter)", te2, {
    "PR-AUC, gradient boosting": lambda i: average_precision_score(y2[i], p2[i]),
    "PR-AUC, burn-rate rule": lambda i: average_precision_score(y2[i], b2[i]),
    "PR-AUC lift over the burn-rate rule": lambda i: average_precision_score(y2[i], p2[i]) - average_precision_score(y2[i], b2[i]),
    "ROC-AUC, gradient boosting": lambda i: roc_auc_score(y2[i], p2[i]),
    "ROC-AUC lift over the burn-rate rule": lambda i: roc_auc_score(y2[i], p2[i]) - roc_auc_score(y2[i], b2[i]),
    "top-10% capture, gradient boosting": lambda i: top_capture(y2[i], p2[i]),
}, lifts=("PR-AUC lift over the burn-rate rule", "ROC-AUC lift over the burn-rate rule"))

ci_table = pd.DataFrame(ci_parts)
ci_table["width"] = ci_table["ci_high"] - ci_table["ci_low"]
print(f"95% confidence intervals (company-clustered bootstrap, {N_BOOT} resamples; share_above_0 = share of resamples where the lift is positive):")
print(ci_table.round(4).to_string(index=False))
lift_rows = ci_table[ci_table["measure"].str.contains("lift")]
print("\nLift intervals (a lift is established only if its whole interval lies above zero):")
for b, m, est, l, h in lift_rows[["block", "measure", "estimate", "ci_low", "ci_high"]].itertuples(index=False):
    print(f"  {b}, {m}: {est:+.3f} [{l:+.3f}, {h:+.3f}] -> " + ("above zero" if l > 0 else "NOT established (the interval reaches zero)"))

fig, ax = plt.subplots(figsize=(7.5, 0.42 * len(ci_table) + 1))
lab = [f"{b.split(' (')[0]} {'(fd)' if 'filing' in b else ''}: {m}" for b, m in zip(ci_table["block"], ci_table["measure"])]
ax.errorbar(ci_table["estimate"], range(len(ci_table)), xerr=[ci_table["estimate"] - ci_table["ci_low"], ci_table["ci_high"] - ci_table["estimate"]],
            fmt="o", color="#0F766E", ecolor="#94A3B8", capsize=3)
ax.axvline(0, color="grey", lw=0.8); ax.set_yticks(range(len(ci_table))); ax.set_yticklabels(lab, fontsize=8); ax.invert_yaxis()
ax.set_title("Point estimates with 95% intervals (company-clustered bootstrap)"); plt.tight_layout(); plt.show()

### Cell 20: decision thresholds chosen on validation data
**Why.** A probability is not a decision. Someone has to say "flag every filing above *t*", and *t* decides how many false alarms and how many missed losses there will be. Choosing *t* on the test period would make the test score flattering, so every threshold here is chosen on the **validation** predictions only and then applied unchanged to the test period.

**The policies.**
- **Best F1 on validation:** the balance of precision and recall.
- **A precision target** (S1: 90%): the most losses found while 9 in 10 flags are right.
- **A recall target** (S2: 80%): for a rare distress event, catch most of them and accept more false alarms.
- **A review list** of fixed size (the riskiest 10% for S1, 5% for S2), for when the number of filings someone can read is the limit.
- **Default 0.5** shown for reference: the probabilities are calibrated, but 0.5 is rarely the right cut-off for a rare event.

**Comparators.** Persistence ("flag if a loss now") for S1, and the burn-rate rule flagging the **same number** of test filings for S2, so precision and recall are compared at the same workload. The last lines check that the default policy behaves on test as it did on validation. Which policy to use is a business choice (the cost of a missed event against the cost of a false alarm); `S1_POLICY` and `S2_POLICY` set the defaults the model card reports.

In [ ]:
# ---- settings: which policy the model card lists as the default (a business choice, not a modelling one)
S1_TARGET_PRECISION = 0.90     # a flagged loss should be right at least 90% of the time
S1_CAPACITY = 0.10             # a review list: the riskiest 10% of filings
S2_TARGET_RECALL = 0.80        # S2 is a screen for a rare event: catch at least 80% of them
S2_CAPACITY = 0.05             # a review list: the riskiest 5% of positive-equity filings
S1_POLICY = "best F1 on validation"
S2_POLICY = f"catch {S2_TARGET_RECALL:.0%} of events on validation"

def flag_stats(y, flag):
    y = np.asarray(y).astype(int); flag = np.asarray(flag, dtype=bool)
    tp = int((flag & (y == 1)).sum()); fp = int((flag & (y == 0)).sum()); fn = int((~flag & (y == 1)).sum())
    prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
    return dict(flag_rate=flag.mean(), precision=prec, recall=rec, f1=2 * prec * rec / max(prec + rec, 1e-12), tp=tp, fp=fp, fn=fn, tn=len(y) - tp - fp - fn)

def thr_best_f1(y, s):
    p, r, t = precision_recall_curve(y, s)
    f = 2 * p[:-1] * r[:-1] / np.maximum(p[:-1] + r[:-1], 1e-12)
    return float(t[np.argmax(f)])
def thr_min_precision(y, s, target):      # the LOWEST threshold that still reaches the target precision = the most recall at that precision
    p, r, t = precision_recall_curve(y, s)
    ok = np.where(p[:-1] >= target)[0]
    return float(t[ok[0]]) if len(ok) else float("inf")
def thr_min_recall(y, s, target):         # the HIGHEST threshold that still reaches the target recall = the fewest flags for that recall
    p, r, t = precision_recall_curve(y, s)
    ok = np.where(r[:-1] >= target)[0]
    return float(t[ok[-1]]) if len(ok) else float("-inf")
def thr_top(s, share):
    return float(np.quantile(s, 1 - share))

def policy_table(policies, yv, sv, yt, st, rule_test=None):
    rows = []
    for name, thr in policies.items():
        v, t = flag_stats(yv, sv >= thr), flag_stats(yt, st >= thr)
        row = dict(policy=name, threshold=thr, valid_flag_rate=v["flag_rate"], valid_precision=v["precision"], valid_recall=v["recall"],
                   test_flag_rate=t["flag_rate"], test_precision=t["precision"], test_recall=t["recall"], test_f1=t["f1"])
        if rule_test is not None:                # the comparator rule flagging the SAME number of test rows
            n_flag = int((st >= thr).sum())
            rf = np.zeros(len(yt), dtype=bool); rf[np.argsort(-rule_test, kind="mergesort")[:n_flag]] = True
            r_ = flag_stats(yt, rf)
            row.update(rule_precision_same_rate=r_["precision"], rule_recall_same_rate=r_["recall"])
        rows.append(row)
    return pd.DataFrame(rows)

# ---- S1: next-quarter loss (all thresholds come from the VALIDATION predictions of the Cell 8 model)
sv1, st1 = pred[("gradient boosting", "valid")], pred[("gradient boosting", "test")]
pers_v, pers_t = flag_stats(yva, va["f_loss_now"] == 1), flag_stats(yte, te["f_loss_now"] == 1)
s1_policies = {"default 0.5 (no choice made)": 0.5,
               "best F1 on validation": thr_best_f1(yva, sv1),
               f"precision {S1_TARGET_PRECISION:.0%} on validation": thr_min_precision(yva, sv1, S1_TARGET_PRECISION),
               f"review list: riskiest {S1_CAPACITY:.0%}": thr_top(sv1, S1_CAPACITY),
               "same flag rate as persistence": thr_top(sv1, pers_v["flag_rate"])}
thr_s1 = policy_table(s1_policies, yva, sv1, yte, st1)
print("S1 decision policies (threshold chosen on VALIDATION, then applied unchanged to TEST):")
print(thr_s1.round(3).to_string(index=False))
print(f"Persistence ('flag if a loss now'), for comparison: test flag rate {pers_t['flag_rate']:.3f}, precision {pers_t['precision']:.3f}, "
      f"recall {pers_t['recall']:.3f}, F1 {pers_t['f1']:.3f}")

# ---- S1 transition: among companies profitable now, flag those likely to turn to a loss
pv_sel, pt_sel = (va["f_loss_now"] == 0).values, (te["f_loss_now"] == 0).values
tr_pol = {"best F1 on validation": thr_best_f1(yva[pv_sel], sv1[pv_sel]),
          f"review list: riskiest {S1_CAPACITY:.0%} of profitable-now rows": thr_top(sv1[pv_sel], S1_CAPACITY)}
thr_tr = policy_table(tr_pol, yva[pv_sel], sv1[pv_sel], yte[pt_sel], st1[pt_sel])
print("\nTransition policies (profitable-now rows only; persistence flags none of them, so its recall here is 0):")
print(thr_tr.round(3).to_string(index=False))

# ---- S2: negative equity next quarter (thresholds from the VALIDATION predictions of the main S2 model)
sv2, st2 = pred2[(S2_MAIN, "valid")], pred2[(S2_MAIN, "test")]
s2_policies = {f"catch {S2_TARGET_RECALL:.0%} of events on validation": thr_min_recall(yva2, sv2, S2_TARGET_RECALL),
               "best F1 on validation": thr_best_f1(yva2, sv2),
               f"review list: riskiest {S2_CAPACITY:.0%}": thr_top(sv2, S2_CAPACITY),
               "default 0.5 (no choice made)": 0.5}
thr_s2 = policy_table(s2_policies, yva2, sv2, yte2, st2, rule_test=S2_BASELINES["burn rate (fewest quarters of cushion)"](te2))
print(f"\nS2 decision policies (validation has {int(yva2.sum())} events, test {int(yte2.sum())}); "
      "rule_* = the burn-rate rule flagging the same number of test rows:")
print(thr_s2.round(3).to_string(index=False))

# ---- does the chosen policy behave on TEST the way it did on VALIDATION?
for label, tab, pol, yt, st in (("S1", thr_s1, S1_POLICY, yte, st1), ("S2", thr_s2, S2_POLICY, yte2, st2)):
    r = tab.set_index("policy").loc[pol]
    cm = flag_stats(yt, st >= r["threshold"])
    print(f"\n{label} default policy '{pol}': threshold {r['threshold']:.3f}")
    print(f"  validation precision {r['valid_precision']:.3f}, recall {r['valid_recall']:.3f}  ->  test precision {r['test_precision']:.3f}, recall {r['test_recall']:.3f}"
          f"  (change {r['test_precision'] - r['valid_precision']:+.3f} / {r['test_recall'] - r['valid_recall']:+.3f})")
    print(f"  test confusion matrix: flagged and right {cm['tp']:,} | flagged but wrong {cm['fp']:,} | missed {cm['fn']:,} | correctly not flagged {cm['tn']:,}")

### Cell 21: later-period, unseen-company and external tests
**Why.** Every score so far comes from the same 14 quarter files. Two questions remain: does the model keep working **further away in time** from its training data, and does it work on **companies it has never seen**?

**What.**
- **(a) Later periods.** One model is trained only on answers public before 1 January 2025 and then never updated. It is scored on every later filing quarter, next to the model retrained every quarter (Cell 18) and persistence. The gap between the two models is the cost of not retraining.
- **(b) Unseen companies.** Test filings from companies with no training row at all (new listings, new SPACs, first-time filers), against the companies the model has seen.
- **(c) External check.** Filings whose outcome was published **after** this version's cut-off (`EXTERNAL_AFTER`, 30 June 2026), scored by a model trained only on what was public by then. With the current dataset there are no such filings yet; the cell runs this test automatically once a newer version of the dataset (quarter files after 2026q2) is attached.

**Status.** (a) and (b) test generalisation beyond the training window *within* this dataset. A test on data from outside this dataset window can only be run when newer DERA quarters are published and added, which is why (c) is a ready-made hook rather than a result.

In [ ]:
EXTERNAL_AFTER = pd.Timestamp("2026-06-30")    # the last filing date of the data this version was evaluated on (quarter files up to 2026q2)

# ---- (a) a model frozen at the validation date (trained only on answers public before it), scored on every later filing quarter
hz = fd_later.assign(p=fd_frozen.loc[fd_later.index, "gradient boosting"].values, filed_q=S.loc[fd_later.index, "filed_q"])
horizon = []
for q, g_ in hz.groupby("filed_q"):
    yy = g_["loss_next"].values
    if len(g_) < ROLLING_MIN_TEST or len(np.unique(yy)) < 2:
        continue
    wf = roll_fd.set_index("filing_quarter")["gbm_roc_auc"].get(str(q), np.nan)
    horizon.append(dict(filing_quarter=str(q), quarters_after_freeze=(q - FD_VALID_START.to_period("Q")).n, rows=len(g_), loss_rate=yy.mean(),
                        frozen_gbm_roc_auc=roc_auc_score(yy, g_["p"]), walk_forward_gbm_roc_auc=wf,
                        persistence_roc_auc=roc_auc_score(yy, g_["f_loss_now"])))
horizon = pd.DataFrame(horizon)
horizon["frozen_lift"] = horizon["frozen_gbm_roc_auc"] - horizon["persistence_roc_auc"]
horizon["cost_of_not_retraining"] = horizon["walk_forward_gbm_roc_auc"] - horizon["frozen_gbm_roc_auc"]
print(f"(a) Later-period test: one model trained on answers public before {FD_VALID_START.date()}, never updated, scored on each later filing quarter")
print(horizon.round(4).to_string(index=False))
print(f"The frozen model beats persistence in {int((horizon['frozen_lift'] > 0).sum())} of {len(horizon)} later quarters; its ROC-AUC goes from "
      f"{horizon['frozen_gbm_roc_auc'].iloc[0]:.3f} (first quarter) to {horizon['frozen_gbm_roc_auc'].iloc[-1]:.3f} "
      f"({int(horizon['quarters_after_freeze'].iloc[-1])} quarters after the freeze). Mean cost of not retraining: {horizon['cost_of_not_retraining'].mean():+.4f} ROC-AUC.")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(horizon["filing_quarter"], horizon["frozen_gbm_roc_auc"], "o-", color="#B45309", label="frozen at the validation date")
ax.plot(horizon["filing_quarter"], horizon["walk_forward_gbm_roc_auc"], "o-", color="#0F766E", label="retrained every quarter (Cell 18)")
ax.plot(horizon["filing_quarter"], horizon["persistence_roc_auc"], "s--", color="#94A3B8", label="persistence")
ax.set_ylabel("ROC-AUC"); ax.set_title("Later-period test: does the model age?"); ax.legend(); plt.xticks(rotation=45); plt.tight_layout(); plt.show()

# ---- (b) companies the model has never seen: no training row at all
def unseen_rows(train_df, test_df, p, protocol):
    out = []
    seen = test_df["cik"].isin(set(train_df["cik"]))
    for label, sel in (("companies seen in training", seen), ("NEW companies (no training row)", ~seen)):
        yy = test_df.loc[sel, "loss_next"].values
        if sel.sum() >= 100 and len(np.unique(yy)) == 2:
            out.append(dict(protocol=protocol, group=label, rows=int(sel.sum()), companies=test_df.loc[sel, "cik"].nunique(), loss_rate=yy.mean(),
                            gbm_roc_auc=roc_auc_score(yy, p[sel.values]), persistence_roc_auc=roc_auc_score(yy, test_df.loc[sel, "f_loss_now"])))
        else:
            out.append(dict(protocol=protocol, group=label, rows=int(sel.sum()), companies=test_df.loc[sel, "cik"].nunique()))
    return out
unseen = pd.DataFrame(unseen_rows(tr, te, pred[("gradient boosting", "test")], "label quarter (Cell 8 model)") +
                      unseen_rows(fd_train_t, fd_test, fd_retrained.loc[fd_test.index, "gradient boosting"].values, "filing date (Cell 18 model)"))
if "gbm_roc_auc" in unseen:
    unseen["lift"] = unseen["gbm_roc_auc"] - unseen["persistence_roc_auc"]
print("\n(b) Unseen-company test (TEST period):")
print(unseen.round(4).to_string(index=False))

# ---- (c) the external check: filings whose outcome was NOT public when this version was evaluated
S["label_delay"] = (S["label_known_date"] - S["next_period"]).dt.days
ext_train = S[S["label_known_date"] <= EXTERNAL_AFTER]
ext = S[(S["label_known_date"] > EXTERNAL_AFTER) & S["label_delay"].between(0, MAX_DAYS_TO_FILE)]
print(f"\n(c) External check. Newest filing in the attached data: {D['filed_date'].max().date()}; cut-off of this version: {EXTERNAL_AFTER.date()}")
if len(ext) >= ROLLING_MIN_TEST and ext["loss_next"].nunique() == 2:
    p_ext, = fit_predict(ext_train, [ext])
    ye = ext["loss_next"].values
    external_result = dict(rows=len(ext), companies=ext["cik"].nunique(), loss_rate=ye.mean(), gbm_roc_auc=roc_auc_score(ye, p_ext),
                           persistence_roc_auc=roc_auc_score(ye, ext["f_loss_now"]), gbm_pr_auc=average_precision_score(ye, p_ext))
    print("Model trained only on answers public by the cut-off, scored on outcomes published after it:")
    print(pd.Series(external_result).round(4).to_string())
else:
    external_result = None
    print(f"Not run: {len(ext)} filings have an outcome published after {EXTERNAL_AFTER.date()} (at least {ROLLING_MIN_TEST} are needed).")
    print("To run it, attach a newer version of the dataset (quarter files after 2026q2) and re-run the notebook. This cell then trains on the")
    print("answers public by the cut-off and scores only outcomes published later: a test on data this version has never seen.")

### Cell 22: model card
A short, standard summary of what the models are for, how well they work (with the intervals from Cell 19 and the thresholds from Cell 20), where they fail and what the risks are. It is **generated from the variables of this run**, so every number matches the outputs above, and it is saved as `model_card.md` in the Output tab together with the Part 3 tables.

In [ ]:
from IPython.display import display, Markdown

def ci_txt(block, measure):
    r = ci_table[(ci_table["block"] == block) & (ci_table["measure"] == measure)].iloc[0]
    return f"{r['estimate']:.3f} (95% CI {r['ci_low']:.3f} to {r['ci_high']:.3f})"
pol1 = thr_s1.set_index("policy").loc[S1_POLICY]; pol2 = thr_s2.set_index("policy").loc[S2_POLICY]
fund = pd.DataFrame(res_).set_index("group")
size_t = slice_table(te_, pred[("gradient boosting", "test")], "size_band").set_index("slice")
n_unseen = unseen[unseen["group"].str.startswith("NEW")]
cal_gap = float((cal_t["mean_predicted"] - cal_t["actual_loss_rate"]).abs().max())
freeze_txt = f"{FD_VALID_START.day} {FD_VALID_START:%B %Y}"
ext_txt = (f"run on {external_result['rows']:,} filings: ROC-AUC {external_result['gbm_roc_auc']:.3f} vs persistence {external_result['persistence_roc_auc']:.3f}"
           if external_result else "not yet possible: the newest data in this version end in 2026q2 (Cell 21 runs it automatically on a newer dataset version)")

s2_pr = ci_table[(ci_table["block"] == "S2 test (label quarter)") & (ci_table["measure"] == "PR-AUC lift over the burn-rate rule")].iloc[0]
s2_status = ("its PR-AUC lead over the burn-rate rule is established on this test period" if s2_pr["ci_low"] > 0 else
             f"its PR-AUC lead over the one-line burn-rate rule ({s2_pr['estimate']:+.3f}) is not statistically established "
             f"(95% CI {s2_pr['ci_low']:+.3f} to {s2_pr['ci_high']:+.3f}, {int(yte2.sum())} test events)")
card = f"""# Model card: next-quarter loss forecast (S1) and exploratory negative-equity screen (S2)
*Generated by Cell 22 from the variables of this run, so every number matches the outputs above. A copy is saved as `model_card.md` in the Output tab.*

## Model details
- **S1:** gradient boosting (`HistGradientBoostingClassifier`, Cell 8 settings) on {len(FEATURES)} features from a company's latest 10-K or 10-Q; output: the probability of a net loss in the next true quarter (75 to 110 days later).
- **S2:** the same recipe (`fit_predict`, Cell 12b) with the target "total equity negative at the next true quarter", for companies with positive equity now; {len(S2_FEATURES)} features.
- **Data:** SEC Financial Statement Data Sets (DERA), dataset `sec-10k-10q-wide-v3` (ETL v3.2), filings from {S['filed_date'].min():%b %Y} to {S['filed_date'].max():%b %Y}.
- **Training rows:** S1 {len(tr):,} filings from {tr['cik'].nunique():,} companies (label quarters up to {TRAIN_END}); S2 {len(tr2):,} filings, {int(ytr2.sum())} events.
- **Leakage controls:** time split, clip limits and every fitted transform learned on training rows only, no amendment flag and no EPS, and a strictly prospective re-test by filing date (Cell 18).

## Status of the evidence
- **S1** is the main model. Its settings were fixed before the test period was scored, and every later choice (features, archetype, cheaper model, thresholds) was made on validation data.
- **S2 is exploratory:** {s2_status}. Use it as a promising screen to prioritise reading, not as a proven improvement over the rule.
- **The test period is not a pristine final test.** It is also reported in the ablation and comparison tables (Cells 12c, 14, 15, 16b), so it has been looked at more than once. An evaluation on data never seen during this work needs newer SEC filings (Cell 21).

## Intended use
- Ranking or flagging filings for **review**: which companies to read first, a watchlist, research on earnings persistence and balance-sheet distress.
- Users who can read a filing and check the flag: analysts, researchers, students.
- Decisions at the level of a **company filing**, refreshed every quarter when new filings arrive.

## Out of scope
- An automatic buy, sell, lending or credit decision on its own. The model sees filings only: no prices, news or macro data.
- Companies the panel cannot score: filers without a true next quarter (irregular or changed fiscal calendars), assets below USD {MIN_ASSETS:,.0f} for the ratio features, or filings more than {MAX_DAYS_TO_FILE} days late.
- Private-credit funds and BDC-style filers (weak, see below), and anything outside SEC 10-K/10-Q filers.

## Performance (test period, with 95% company-clustered bootstrap intervals, Cell 19)
| Measure | Value |
|---|---|
| S1 ROC-AUC, gradient boosting | {ci_txt('S1 test (label quarter)', 'ROC-AUC, gradient boosting')} |
| S1 ROC-AUC, persistence baseline | {ci_txt('S1 test (label quarter)', 'ROC-AUC, persistence')} |
| S1 lift over persistence (ROC-AUC) | +{ci_txt('S1 test (label quarter)', 'ROC-AUC lift over persistence')} |
| S1 PR-AUC, gradient boosting | {ci_txt('S1 test (label quarter)', 'PR-AUC, gradient boosting')} |
| S1 transition (profitable now) ROC-AUC | {ci_txt('S1 test (label quarter)', 'transition ROC-AUC (profitable now)')} |
| S1 ROC-AUC, filing-date protocol (Cell 18) | {ci_txt('S1 test (filing date)', 'ROC-AUC, gradient boosting')} |
| S2 PR-AUC, gradient boosting | {ci_txt('S2 test (label quarter)', 'PR-AUC, gradient boosting')} |
| S2 PR-AUC, burn-rate rule | {ci_txt('S2 test (label quarter)', 'PR-AUC, burn-rate rule')} |
| S2 PR-AUC lift over the burn-rate rule | {ci_txt('S2 test (label quarter)', 'PR-AUC lift over the burn-rate rule')} |
| S2 top-10% capture | {ci_txt('S2 test (label quarter)', 'top-10% capture, gradient boosting')} |

Rolling checks: S1 beats persistence in {int((roll['roc_auc_lift'] > 0).sum())} of {len(roll)} label quarters (Cell 12b) and {int((roll_fd['roc_auc_lift'] > 0).sum())} of {len(roll_fd)} filing quarters (Cell 18). Calibration on test (Cell 9): across the ten probability buckets, predicted and actual loss rates differ by at most {cal_gap:.3f}.

## Decision thresholds (chosen on validation only, Cell 20)
| Use | Policy | Threshold | Test precision | Test recall | Test flag rate |
|---|---|---|---|---|---|
| S1 next-quarter loss | {S1_POLICY} | {pol1['threshold']:.3f} | {pol1['test_precision']:.3f} | {pol1['test_recall']:.3f} | {pol1['test_flag_rate']:.1%} |
| S2 negative equity | {S2_POLICY} | {pol2['threshold']:.3f} | {pol2['test_precision']:.3f} | {pol2['test_recall']:.3f} | {pol2['test_flag_rate']:.1%} |

Pick the policy from the cost of a missed event against the cost of a false alarm; Cell 20 lists the alternatives (a precision target, a fixed review-list size).

## Generalisation (Cell 21)
- **Later periods:** a model frozen on {freeze_txt} still beats persistence in {int((horizon['frozen_lift'] > 0).sum())} of {len(horizon)} later filing quarters (up to {int(horizon['quarters_after_freeze'].max())} quarters ahead); retraining every quarter adds {horizon['cost_of_not_retraining'].mean():+.3f} ROC-AUC on average.
- **New companies:** {(f"{int(n_unseen['rows'].iloc[0]):,} test filings from companies with no training row score ROC-AUC {n_unseen['gbm_roc_auc'].iloc[0]:.3f}" if len(n_unseen) and 'gbm_roc_auc' in n_unseen and pd.notna(n_unseen['gbm_roc_auc'].iloc[0]) else "too few test filings from new companies to score")} (label-quarter protocol).
- **External data:** {ext_txt}.

## Known limitations and failure modes
- **Fund-like filers:** test ROC-AUC {fund.loc['fund-like', 'roc_auc'] if 'fund-like' in fund.index else float('nan'):.3f} against {fund.loc['other', 'roc_auc'] if 'other' in fund.index else float('nan'):.3f} for other companies (Cell 12d): their losses follow market valuations that filings do not foreshadow.
- **Shared shocks:** anything that hits a whole group at once (credit spreads, a recession) is invisible until it appears in a filing.
- **One-off corporate actions** (large buybacks, impairments, acquisitions) drive the biggest S2 misses (Cell 16c).
- **SPAC equity:** redeemable shares sit outside equity, so SPACs often show negative equity by convention; S2 reports them as a separate slice.
- **Size:** test ROC-AUC by asset tercile: {', '.join(f'{k} {v:.3f}' for k, v in size_t['roc_auc'].items())} (Cell 10).
- **Short history:** about two years of training labels; seasonality is learned from at most two cycles.

## Risks and how to manage them
- **Over-reliance:** a calibrated probability still means some flagged companies stay profitable. Use the flag to prioritise reading, not to replace it.
- **Staleness:** retrain every quarter on the answers public at that date (the Cell 18 walk-forward); watch the rolling ROC-AUC, PR-AUC and calibration, and stop using the model if it no longer beats persistence.
- **Data errors:** Cell 4 must print `OK`; a `WARNING` means an old or inconsistent dataset version. Read the largest errors (Cells 12 and 16c) after every run.
- **Unequal accuracy:** check the slice tables (Cells 10 and 16b) before using the scores for one sector or size group.
- **No personal data:** the inputs are public company filings.
"""
display(Markdown(card))

OUT3 = "/kaggle/working"
with open(f"{OUT3}/model_card.md", "w") as fh:
    fh.write(card)
fd_compare.round(5).to_csv(f"{OUT3}/part3_filing_date_validation.csv", index=False)
roll_fd.round(5).to_csv(f"{OUT3}/part3_walk_forward_by_filing_quarter.csv", index=False)
ci_table.round(5).to_csv(f"{OUT3}/part3_confidence_intervals.csv", index=False)
pd.concat([thr_s1.assign(task="S1"), thr_tr.assign(task="S1 transition"), thr_s2.assign(task="S2")]).round(5).to_csv(f"{OUT3}/part3_thresholds.csv", index=False)
horizon.round(5).to_csv(f"{OUT3}/part3_later_period.csv", index=False)
unseen.round(5).to_csv(f"{OUT3}/part3_unseen_companies.csv", index=False)
print(f"model card: {card.count(chr(10))} lines -> {OUT3}/model_card.md")
print("saved:", ", ".join(sorted(f for f in os.listdir(OUT3) if f.startswith(("part3_", "model_card")))))

---
### How to read the results

| Output | What good looks like | What to do if not |
|---|---|---|
| Cell 4 | `OK`: ETL-derived quarters agree with the period chain | `WARNING` means an old dataset version is attached: attach ETL v3.2 |
| Cell 6 | Each split has thousands of rows; the loss rate is near 50% | Change `TRAIN_END` / `VALID_END` |
| Cell 7 | Persistence is strong. Its ROC-AUC and PR-AUC are the bar | None, this is the baseline |
| Cell 8 | GroupKFold scores (preparation refitted inside each fold) close to the validation scores | A big gap means the split or a feature leaks |
| Cell 9 | Positive lift over persistence on **both** validation and test, and a useful ranking in the transition view | Little or no lift is a real finding: report it as such |
| Cell 10 | Skill in the operating-company slices, not only in SPAC/shell slices | Consider separate models by segment |
| Cell 12 | Errors that are surprises, not data glitches | Fix the glitches in the ETL, then re-run |
| Cell 12b | Positive lift over persistence in every rolling quarter | A quarter where the model loses to persistence is the place to look |
| Cell 12c | Each feature group adds something; seasonality (`ni_a3`) matters | A group whose removal costs nothing can be dropped |
| Cell 12d | Fund-like filers are a small, clearly separated group | A large group of shared-shock filers would call for its own model or extra inputs |
| Cell 14 (U1) | Archetypes with clearly different profiles and loss rates that stay similar across splits | If the archetype adds < 0.002 test ROC-AUC, keep it as a description, not a feature |
| Cell 15 | A small model that keeps most of the lift | If even the full model's lift is small, publish the small one |
| Cell 16 (S2) | Positive PR-AUC lift over the *best* rule on test and in most rolling quarters, and skill outside SPACs | If the burn-rate rule wins, report that: a one-line rule is the honest answer. Check the interval in Cell 19 before calling any lead proven |
| Cell 18 | Filing-date scores within about 0.01 of the label-quarter scores, positive lift in (almost) every filing quarter | A large drop means the label-quarter split was optimistic: quote the filing-date numbers instead |
| Cell 19 | Lift intervals entirely above zero | An interval that reaches zero means the lift is not established |
| Cell 20 | Test precision and recall close to their validation values for the chosen policy | A large change means the threshold does not transfer: re-choose it on more recent data |
| Cell 21 | The frozen model still beats persistence in later quarters; new companies score close to known ones | Fast decay means the model must be retrained every quarter |
| Cell 22 | A model card whose numbers match the outputs above | Re-run the notebook from the top |

### How the test period was used
- **Fixed before the test period was scored:** the S1 model and its settings (Cell 8), the features, the split dates and the filters.
- **Chosen on validation data only:** the feature ranking (Cell 11), the archetype verdict (Cell 14), the cheaper model (Cell 15) and every decision threshold (Cell 20).
- **Reported on the test period, without changing any choice:** the ablation tables (Cells 12c, 14 and 16b), the cheaper-model table (Cell 15), the slices and the error lists.
- **What this means:** the headline test scores are honest out-of-time scores for a model fixed in advance, but the test period has been viewed more than once, so it is not a pristine, single-use final test. The fully independent check is the external evaluation in Cell 21, which needs SEC filings published after this dataset ends.

### Limitations
- **S2 is exploratory.** With 141 test events, the interval for its PR-AUC lead over the burn-rate rule includes zero (Cell 19). It is a promising screening model, not a proven improvement.
- **Filings only.** Nothing here sees prices, credit spreads or the economy, so shocks shared across a group (for example private-credit funds in 2026Q1, Cell 12d) and one-off corporate actions (the S2 misses, Cell 16c) are invisible until they appear in a filing.
- **Short history.** 14 quarterly files give about two years of training labels. Seasonality is learned from at most two cycles.
- **No external test yet.** Every test, including the later-period ones in Cell 21, comes from this dataset window. The external check in Cell 21 needs quarter files after 2026q2.
- **Filing dates are rebuilt** as period end + `days_to_file`. An amended or re-filed report can make an answer public later than this date suggests.
- **Fiscal calendars.** The "true next quarter" rule (75 to 110 days) drops companies that change fiscal year or file irregularly. They are not scored at all.
- **SPAC equity.** Many SPACs report negative equity because redeemable shares sit outside equity. S2 reports them as a separate slice for that reason.

### For forks and edits
The notebook is built to be re-run and extended. Some starting points:

**Change a setting.** Every setting sits at the top of the cell that uses it: the split dates and filters in Cell 1 (`TRAIN_END`, `VALID_END`, `GAP_RANGE`, `MIN_ASSETS`), the rolling-origin window in Cell 12b, the archetype range and rule in Cell 14 (`ARCH_K_RANGE`, `ARCH_K`, `ARCH_MIN_SHARE`), the cheaper-model size in Cell 15 (`TOP_K`), the forecast dates in Cell 18, the number of bootstrap resamples in Cell 19 (`N_BOOT`), and the decision policies in Cell 20 (`S1_POLICY`, `S2_POLICY` and their targets). Everything downstream reads these names, so the tables, the headline numbers and the model card update on the next run.

**Add newer data.** When DERA publishes quarters after 2026q2, publish a new version of the dataset and attach it. The quarter files are found by name, Cell 21's external check then scores outcomes this version has never seen, and Cells 12b and 18 extend their rolling tests automatically.

**Ideas worth building on**
- Market or macro inputs (for example a credit-spread index by quarter) for the fund-like filers, tested with the Cell 12c ablation.
- Announced corporate actions (buybacks, impairments, acquisitions) for S2, whose largest misses come from them (Cell 16c).
- More test events for S2 (pooled rolling quarters or a longer window), so that its PR-AUC lead over the burn-rate rule can be confirmed or rejected (Cell 19).
- A single-use final test: freeze every choice on validation data, then score newer SEC filings exactly once (Cell 21 is built for this).
- Separate models for operating companies, SPACs and fund-like filers, compared with the single model on the same slices.
- A monitoring cell that tracks the rolling ROC-AUC, PR-AUC and calibration each quarter, using the walk-forward loop in Cell 18.

**Keep the three rules.** Fit every learned quantity on training rows only (`fit_predict` and `prep_rows` already do), report every score next to the persistence rule, and evaluate on later filings than the ones used for training.

### Data-quality decisions behind the panel
These choices were made while building the panel. Each one is checked or printed by the cell named.

| Decision | Cell | Why |
|---|---|---|
| The standalone quarterly profit is **rebuilt from the cumulative figures by period order** (the previous row must end about one quarter earlier and cover one quarter fewer) | 4 | Chaining by fiscal-year label can subtract a row from the wrong fiscal year. In ETL versions before 3.2, NVIDIA's 10-K for 2025-01-31 showed a standalone quarter of USD 55.4 billion: its fiscal-2025 annual profit (USD 72.9 billion) minus its fiscal-2024 *nine-month* profit (USD 17.5 billion). The true fourth quarter was USD 22.1 billion. Cell 4 prints `OK` when the attached dataset agrees with the rebuild |
| A dataset **version check** by column name, and a clear `OK` or `WARNING` | 2, 4 | An older dataset version cannot pass unnoticed |
| **One filing per company and quarter** (the latest one filed) | 3 | A duplicate quarter sits 0 days from its twin, so its neighbours fall outside the 75-110 day rule and get a blank or wrong previous or next quarter |
| Filings are kept only with a **filing delay of 0 to 365 days** and no negative value in a tag that can never be negative | 6 | A negative delay (the raw data go down to −238 days) is impossible; delays of years (up to 4,018 days) are catch-up filings, where "the information available at filing" is not what the model is meant to use |
| **EPS is not a feature** | 5 | The raw EPS values contain scale errors (for example −713,794 and +1.85 billion per share) that cannot be rebuilt from other columns; profit scaled by assets carries the same information |
| **The amendment flag is not a feature** | 5 | Whether a filing gets amended is only known later (a mild look-ahead); removing it changed neither the validation nor the test ROC-AUC (0.0000) |
| **Clipping limits, medians, scalers and cluster centres come from the training rows only** | 8, 12b, 14, 16b | Limits computed on all rows would pass information from the validation and test periods into training |
| A **narrow fund-like name pattern** with an exclusion list | 12d | A broad pattern also caught captive finance arms, banks, REITs, royalty trusts and tax-credit funds, which are not private-credit funds or BDCs |